# เฉลยแบบฝึกหัด บทที่ 2 — ระเบียบวิธีประเมินผลระบบผู้แนะนำ
## 12 ข้อ + โจทย์ท้าทาย · พร้อมคำอธิบายทีละข้อ

---

> ⚠️ **สมุดเล่มนี้คือเฉลย** ควรลองทำใน `Lab2_Exercises.ipynb` ให้ครบก่อนเปิดดู
> เพราะประโยชน์ของแบบฝึกหัดอยู่ที่ "ตอนที่คำนวณแล้วผิด" ไม่ใช่ตอนที่อ่านคำตอบที่ถูก

### วิธีอ่านเฉลย

แต่ละข้อมีสามส่วน

1. **โจทย์** — เหมือนในสมุดแบบฝึกหัดทุกประการ
2. **โค้ดเฉลย + ผลตรวจ** — รันได้ทันที ทุกข้อต้องขึ้น ✅ ครบ
3. **📝 ทำไมจึงเป็นเช่นนั้น** — คำอธิบายว่าโจทย์ข้อนั้นกำลังสอนอะไร และจุดที่พลาดกันบ่อยที่สุดคืออะไร

### แผนผังเฉลย

| ข้อ | เรื่อง | ตรงกับหัวข้อในตำรา | จุดที่พลาดบ่อยที่สุด |
|---|---|---|---|
| 1 | MAE และ RMSE | 2.1 | ลืมถอดรากที่สองใน RMSE |
| 2 | MAE เท่ากันแต่ RMSE ต่างกัน | 2.1 | คิดว่า MAE เท่ากัน = ความผิดพลาดเหมือนกัน |
| 3 | Precision · Recall · F1 | 2.2 | ใช้ K เป็นตัวส่วนของ Recall |
| 4 | RR และ MRR@K | 2.3.1 | ลืมตัดลิสต์ที่ K ก่อนค้นหา |
| 5 | AP และ MAP@K | 2.3.2 | หารด้วย K แทน min(K, \|R\|) |
| 6 | DCG · IDCG · nDCG@K | 2.3.3 | สร้าง IDCG จากลิสต์ของแบบจำลองแทนลิสต์อุดมคติ |
| 7 | AUC | 2.4 | ตีความว่าเป็น "ความแม่นยำ" |
| 8 | Coverage และ Gini | 2.5.1–2.5.2 | ลืมเรียง และตัดรายการที่ได้ 0 ทิ้ง |
| 9 | Novelty และ LongTail | 2.5.3–2.5.4 | ใช้ ln แทน log₂ |
| 10 | Calibration และ Parity Gap | 2.5.5–2.5.6 | ตกหลุม Leveling Down |
| 11 | Temporal Split และ Leakage | 2.8 | คำนวณสถิติก่อนแบ่งข้อมูล |
| 12 | Sampled Negatives และ Winner Flip | 2.9 | คิดว่าการสุ่มบีบทุกโมเดลเท่ากันจึงยุติธรรม |
| ★ | สร้างแบบจำลองของตัวเอง | 2.6 + 2.10 | ปรับ Primary Metric จนพัง Guardrail |

---
## ส่วนที่ 0 — เตรียมข้อมูลและระบบตรวจคำตอบ (รันก่อนเสมอ)

In [ ]:
# ============================================================================
#  เซลล์เตรียมข้อมูล — รันเซลล์นี้ก่อนทำแบบฝึกหัดทุกข้อ
#  (ไม่ต้องอัปโหลดไฟล์ใด ๆ ข้อมูลทั้งหมดสร้างขึ้นในเซลล์นี้)
# ============================================================================
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
SEED = 2569

# ---------------------------------------------------------------- ระบบตรวจคำตอบ
_SCORE = {}


def check(label, got, expected, tol=1e-4, hint=""):
    """เทียบคำตอบกับค่าอ้างอิง  รองรับทั้งตัวเลขเดี่ยว ลิสต์ และ dict"""
    def close(a, b):
        if isinstance(b, dict):
            return set(a) == set(b) and all(close(a[k], b[k]) for k in b)
        if isinstance(b, (list, tuple, np.ndarray)):
            a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
            return a.shape == b.shape and np.allclose(a, b, atol=tol)
        if isinstance(b, (set, frozenset)):
            return set(a) == set(b)
        if isinstance(b, bool):
            return bool(a) is b
        try:
            return abs(float(a) - float(b)) <= tol
        except (TypeError, ValueError):
            return a == b

    try:
        ok = close(got, expected)
    except Exception as e:                       # noqa: BLE001
        ok, got = False, f"<ผิดพลาด: {e}>"
    _SCORE[label] = bool(ok)
    mark = "✅" if ok else "❌"
    def fmt(v):
        if isinstance(v, float):
            return f"{v:.4f}"
        if isinstance(v, (list, tuple, np.ndarray)):
            return "[" + ", ".join(f"{float(x):.4f}" for x in np.asarray(v).ravel()) + "]"
        return str(v)
    print(f"{mark}  {label:<46} ได้ {fmt(got)}"
          + ("" if ok else f"   ควรได้ {fmt(expected)}"))
    if not ok and hint:
        print(f"     💡 {hint}")
    return ok


def show_score():
    """สรุปคะแนนรวมของทั้งสมุด"""
    if not _SCORE:
        print("ยังไม่ได้ตรวจข้อใดเลย")
        return
    n_ok = sum(_SCORE.values())
    n = len(_SCORE)
    print("=" * 66)
    print(f"คะแนนรวม : {n_ok} / {n}  ({n_ok/n*100:.0f}%)")
    print("=" * 66)
    bad = [k for k, v in _SCORE.items() if not v]
    if bad:
        print("ข้อที่ยังไม่ผ่าน :")
        for b in bad:
            print("   -", b)
    else:
        print("ผ่านครบทุกข้อ 🎉")


# ---------------------------------------------------- ข้อมูลชุดที่ 1: ลิสต์อันดับ
# Model A และ B หยิบของถูกจำนวนเท่ากัน (3 ชิ้น) แต่วางคนละตำแหน่ง
RELEVANT = {"i2", "i5", "i9"}
RANK_A = ["i2", "i7", "i5", "i3", "i9", "i1", "i4", "i6", "i8", "i10"]
RANK_B = ["i7", "i3", "i1", "i2", "i4", "i5", "i6", "i8", "i9", "i10"]

# ---------------------------------------------- ข้อมูลชุดที่ 2: ผู้ใช้สามคน
# ใช้ในข้อ 4 และ 5
# ผู้ใช้แต่ละคนมีรายการที่ชอบจริงสามชิ้นเหมือนกันทั้งสองแบบจำลอง (rel_1, rel_2, rel_3)
# Model A และ B ต่างกันเพียง "ตำแหน่ง" ที่วางรายการทั้งสามนั้น
def _mk(user, positions, n=10):
    lst = [None] * n
    for idx, p in enumerate(sorted(positions), start=1):
        lst[p - 1] = f"{user}_rel{idx}"
    fill = 0
    for i in range(n):
        if lst[i] is None:
            fill += 1
            lst[i] = f"{user}_oth{fill}"
    return lst


_POS = {"Emma": ((1, 3, 5), (4, 6, 9)),
        "Liam": ((2, 4, 7), (1, 5, 8)),
        "Noah": ((2, 5, 6), (4, 7, 10))}
THREE_USERS = {}
for _u, (_pa, _pb) in _POS.items():
    THREE_USERS[_u] = {"A": _mk(_u, _pa), "B": _mk(_u, _pb),
                       "relevant": {f"{_u}_rel{i}" for i in (1, 2, 3)}}

# -------------------------------------------- ข้อมูลชุดที่ 3: MovieMini-12
MOVIE_META = pd.DataFrame([
    ("i1",  "The Matrix",    "ไซไฟ",      "ค่าย ก", 400),
    ("i2",  "Inception",     "ไซไฟ",      "ค่าย ก", 150),
    ("i3",  "Interstellar",  "ไซไฟ",      "ค่าย ก",  25),
    ("i4",  "Titanic",       "โรแมนติก",  "ค่าย ข", 300),
    ("i5",  "La La Land",    "โรแมนติก",  "ค่าย ข", 100),
    ("i6",  "The Notebook",  "โรแมนติก",  "ค่าย ง",   8),
    ("i7",  "John Wick",     "แอ็กชัน",   "ค่าย ค", 250),
    ("i8",  "Mad Max",       "แอ็กชัน",   "ค่าย ง",  15),
    ("i9",  "Parasite",      "ระทึกขวัญ", "ค่าย ข",  60),
    ("i10", "Toy Story",     "แอนิเมชัน", "ค่าย ข", 200),
    ("i11", "Coco",          "แอนิเมชัน", "ค่าย ค",  40),
    ("i12", "Spirited Away", "แอนิเมชัน", "ค่าย ง",   2),
], columns=["item_id", "item_name", "genre", "studio", "viewers"])

IDS = list(MOVIE_META.item_id)
GENRE = dict(zip(MOVIE_META.item_id, MOVIE_META.genre))
STUDIO = dict(zip(MOVIE_META.item_id, MOVIE_META.studio))
VIEWERS = dict(zip(MOVIE_META.item_id, MOVIE_META.viewers))
PLATFORM_USERS = 500          # ฐานผู้ใช้ทั้งแพลตฟอร์ม ใช้คำนวณ p(i)

USERS8 = ["Emma", "James", "Olivia", "Michael", "Sophia", "Daniel", "Ava", "Ethan"]
K5 = 5

# Model X : ยอดนิยมชุดเดียวให้ทุกคน | Model Y : เฉพาะบุคคล
MODEL_X = {u: ["i1", "i4", "i7", "i10", "i2"] for u in USERS8}
MODEL_Y = {
    "Emma":    ["i1", "i2", "i3", "i9", "i12"],
    "James":   ["i1", "i2", "i3", "i10", "i5"],
    "Olivia":  ["i4", "i5", "i6", "i11", "i9"],
    "Michael": ["i3", "i7", "i8", "i2", "i1"],
    "Sophia":  ["i4", "i5", "i6", "i10", "i12"],
    "Daniel":  ["i2", "i3", "i9", "i7", "i8"],
    "Ava":     ["i2", "i4", "i6", "i11", "i10"],
    "Ethan":   ["i1", "i5", "i7", "i3", "i11"],
}
USER_GROUP = {"ประวัติยาว": ["Emma", "Olivia", "Sophia", "Daniel"],
              "ประวัติสั้น": ["James", "Michael", "Ava", "Ethan"]}

# เฉลยความเกี่ยวข้อง: เรื่องที่ผู้ใช้ให้คะแนน ≥ 4
_R = np.array([
    [5, 5, 4, 1, np.nan, 1, 4, np.nan, 4, np.nan, np.nan, 3],
    [4, 5, 5, np.nan, 2, np.nan, 3, 4, np.nan, 2, np.nan, np.nan],
    [1, 2, np.nan, 5, 5, 4, np.nan, np.nan, 3, np.nan, 4, 4],
    [np.nan, 3, 4, 2, np.nan, np.nan, 5, 5, 4, np.nan, np.nan, np.nan],
    [2, np.nan, 1, 5, 5, 5, np.nan, 1, np.nan, 4, 5, np.nan],
    [np.nan, 4, 5, 3, 3, np.nan, 2, np.nan, 5, 4, 4, 5],
    [3, 4, np.nan, 4, np.nan, 3, np.nan, np.nan, 5, np.nan, 3, 4],
    [4, np.nan, 3, np.nan, 4, np.nan, 5, 4, np.nan, 5, np.nan, 5]], dtype=float)
RATINGS = pd.DataFrame(_R, index=USERS8, columns=IDS)
RELEVANT8 = {u: {i for i in IDS if RATINGS.loc[u, i] >= 4} for u in USERS8}
PROFILE8 = RELEVANT8            # ใช้ประวัติ = เรื่องที่ชอบ สำหรับข้อ 10

# ------------------------------------------- ข้อมูลชุดที่ 4: RSLab-Eval (ข้อ 11)
def make_interactions(n_users=400, n_items=200, n_weeks=20, seed=SEED):
    """ปฏิสัมพันธ์สังเคราะห์ที่ความนิยมเลื่อนตามเวลา (popularity drift)"""
    rng = np.random.default_rng(seed)
    items = np.array([f"it{n:03d}" for n in range(n_items)])
    base = rng.pareto(1.6, n_items) + 1
    base = base / base.sum()
    drift = rng.normal(0.0, 0.9, n_items)
    rows = []
    for w in range(n_weeks):
        pw = base * np.exp(drift * (w / max(1, n_weeks - 1)))
        pw = pw / pw.sum()
        for _ in range(int(rng.integers(400, 600))):
            u = int(rng.integers(0, n_users))
            i = int(rng.choice(n_items, p=pw))
            rows.append({"user_id": f"u{u:03d}", "item_id": items[i],
                         "rating": float(np.clip(round(rng.normal(3.6, 1.0)), 1, 5)),
                         "timestamp": w * 7 * 86400 + float(rng.uniform(0, 7 * 86400)),
                         "week": w})
    return (pd.DataFrame(rows).sort_values("timestamp")
            .drop_duplicates(subset=["user_id", "item_id"], keep="first")
            .reset_index(drop=True))


INTERACTIONS = make_interactions()

print("เตรียมข้อมูลเสร็จแล้ว\n")
print(f"  RELEVANT     : {sorted(RELEVANT)}")
print(f"  RANK_A       : {RANK_A}")
print(f"  RANK_B       : {RANK_B}")
print(f"  THREE_USERS  : {list(THREE_USERS)} (แต่ละคนมีลิสต์ของ Model A และ B)")
print(f"  MOVIE_META   : {len(MOVIE_META)} เรื่อง | MODEL_X, MODEL_Y : ผู้ใช้ {len(USERS8)} คน K = {K5}")
print(f"  INTERACTIONS : {len(INTERACTIONS):,} แถว "
      f"({INTERACTIONS.user_id.nunique()} ผู้ใช้ × {INTERACTIONS.item_id.nunique()} รายการ, "
      f"{INTERACTIONS.week.nunique()} สัปดาห์)")
print("\nพร้อมทำแบบฝึกหัดแล้ว — เริ่มที่ข้อ 1")

---
## ข้อ 1 — MAE และ RMSE ⭐
**หัวข้อ 2.1**

เขียนฟังก์ชัน `my_mae(y_true, y_pred)` และ `my_rmse(y_true, y_pred)` ด้วยตัวเอง

$$\text{MAE}=\frac{1}{n}\sum_{i}|\hat r_i-r_i| \qquad
\text{RMSE}=\sqrt{\frac{1}{n}\sum_{i}(\hat r_i-r_i)^2}$$

ทดสอบกับข้อมูลของผู้ใช้ Emma

| ภาพยนตร์ | คะแนนจริง | Model A ทาย |
|---|---|---|
| The Matrix | 5 | 4.2 |
| Inception | 5 | 4.6 |
| Interstellar | 4 | 4.3 |
| Titanic | 1 | 2.4 |
| The Notebook | 1 | 1.5 |

จากนั้นตอบคำถาม: **อัตราส่วน RMSE/MAE บอกอะไรเรา**

In [ ]:
# ============================== เฉลยข้อ 1 ==============================
def my_mae(y_true, y_pred) -> float:
    """ค่าเฉลี่ยความคลาดเคลื่อนสัมบูรณ์"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def my_rmse(y_true, y_pred) -> float:
    """รากที่สองของค่าเฉลี่ยความคลาดเคลื่อนกำลังสอง"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# --------------------------- ตรวจคำตอบ ---------------------------
y_true = [5, 5, 4, 1, 1]
y_pred = [4.2, 4.6, 4.3, 2.4, 1.5]

# ตารางตรวจด้วยมือ
print(f"{'รายการ':<16}{'จริง':>7}{'ทาย':>8}{'e':>9}{'|e|':>9}{'e²':>10}")
print("-" * 60)
names = ["The Matrix", "Inception", "Interstellar", "Titanic", "The Notebook"]
for nm, t, p in zip(names, y_true, y_pred):
    e = p - t
    print(f"{nm:<16}{t:>7.1f}{p:>8.2f}{e:>9.2f}{abs(e):>9.2f}{e**2:>10.4f}")
s_abs = sum(abs(p - t) for t, p in zip(y_true, y_pred))
s_sq = sum((p - t) ** 2 for t, p in zip(y_true, y_pred))
print("-" * 60)
print(f"{'ผลรวม':<16}{'':>7}{'':>8}{'':>9}{s_abs:>9.2f}{s_sq:>10.4f}")
print(f"\nMAE  = {s_abs:.2f} / 5 = {my_mae(y_true, y_pred):.4f}")
print(f"RMSE = sqrt({s_sq:.4f} / 5) = sqrt({s_sq/5:.4f}) = {my_rmse(y_true, y_pred):.4f}\n")

check("1.1 MAE ของ Model A", my_mae(y_true, y_pred), 0.6800)
check("1.2 RMSE ของ Model A", my_rmse(y_true, y_pred), 0.7874)
check("1.3 RMSE ≥ MAE เสมอ",
      all(my_rmse(a, b) >= my_mae(a, b) - 1e-12
          for a, b in [(np.random.rand(8) * 5, np.random.rand(8) * 5) for _ in range(200)]),
      True)

print(f"\nอัตราส่วน RMSE/MAE = {my_rmse(y_true, y_pred)/my_mae(y_true, y_pred):.4f}")

### 📝 ข้อ 1 กำลังสอนอะไร

**คำตอบของคำถามท้ายข้อ** อัตราส่วน `RMSE/MAE` บอก *รูปร่างการกระจายของความผิดพลาด* ไม่ใช่ขนาดของมัน

* อัตราส่วน = 1.0000 พอดี → ความผิดพลาดทุกรายการมีขนาดเท่ากันเป๊ะ
* อัตราส่วนสูง → ความผิดพลาดกระจุกอยู่ที่ไม่กี่รายการ (มี outlier)

ในตัวอย่างนี้ได้ 1.1579 ซึ่งมาจาก Titanic ที่ทายพลาดถึง 1.40 ดาวเพียงเรื่องเดียว
ขณะที่อีกสี่เรื่องพลาดไม่เกิน 0.80 ดาว

**จุดที่พลาดกันบ่อยที่สุด** ลืมถอดรากที่สอง — ได้ MSE แทน RMSE ซึ่งหน่วยกลายเป็น "ดาวกำลังสอง"
ตีความไม่ได้และเทียบกับ MAE ไม่ได้เลย

**ทำไม `RMSE ≥ MAE` เสมอ** เป็นผลโดยตรงจากอสมการของเยนเซน (Jensen's inequality)
เพราะฟังก์ชันยกกำลังสองเป็นฟังก์ชันคอนเวกซ์ ความเท่ากันเกิดขึ้นก็ต่อเมื่อ
ความคลาดเคลื่อนทุกตัวมีขนาดเท่ากันพอดี — ซึ่งแทบไม่เกิดในข้อมูลจริง

> **ข้อปฏิบัติ** รายงาน MAE และ RMSE คู่กันเสมอ MAE สรุปขนาดความผิดพลาดโดยเฉลี่ย
> ส่วน RMSE ตรวจจับความเสี่ยงจากความผิดพลาดก้อนใหญ่ — เลือกค่าหลักให้สอดคล้องกับ
> ต้นทุนของความผิดพลาดในงานจริง

---
## ข้อ 2 — ออกแบบกรณีที่ MAE เท่ากันแต่ RMSE ต่างกัน ⭐⭐
**หัวข้อ 2.1**

กำหนดคะแนนจริงของสี่รายการเท่ากับ `[4, 4, 4, 4]` ทั้งหมด

จงหา **ค่าพยากรณ์สองชุด** ที่

* ชุดที่ 1 `pred_even` — ให้ `MAE = 1.0` โดยผิดพลาด **เท่ากันทุกรายการ**
* ชุดที่ 2 `pred_spiky` — ให้ `MAE = 1.0` เท่ากันเป๊ะ แต่ `RMSE` **สูงกว่าชุดแรกอย่างน้อย 1.9 เท่า**

จากนั้นเขียนคำตอบว่าเหตุใด MAE จึงมองไม่เห็นความต่างนี้ (ลงในตัวแปร `answer_2`)

In [ ]:
# ============================== เฉลยข้อ 2 ==============================
y_true_2 = [4, 4, 4, 4]

pred_even = [3.0, 3.0, 3.0, 3.0]       # ผิดทีละ 1.0 ทุกรายการ -> MAE = 1.0, RMSE = 1.0
pred_spiky = [4.0, 4.0, 4.0, 0.0]      # ถูกสามรายการ แต่พลาด 4.0 หนึ่งรายการ -> MAE = 1.0, RMSE = 2.0

answer_2 = ("MAE เฉลี่ยขนาดความผิดพลาดแบบเชิงเส้น ความผิดพลาด 4 ดาวหนึ่งครั้ง "
            "จึงมีค่าเท่ากับความผิดพลาด 1 ดาวสี่ครั้งพอดี  ส่วน RMSE ยกกำลังสองก่อนเฉลี่ย "
            "ความผิดพลาดก้อนใหญ่จึงถูกลงโทษหนักกว่ามาก (4² = 16 เทียบกับ 4 × 1² = 4)")

# --------------------------- ตรวจคำตอบ ---------------------------
m_even, r_even = my_mae(y_true_2, pred_even), my_rmse(y_true_2, pred_even)
m_spiky, r_spiky = my_mae(y_true_2, pred_spiky), my_rmse(y_true_2, pred_spiky)
print(f"ชุดที่ 1  {pred_even}   MAE = {m_even:.4f}  RMSE = {r_even:.4f}")
print(f"ชุดที่ 2  {pred_spiky}   MAE = {m_spiky:.4f}  RMSE = {r_spiky:.4f}")
print(f"อัตราส่วน RMSE ของสองชุด = {r_spiky/r_even:.2f} เท่า\n")

check("2.1 MAE ของชุดที่ 1 เท่ากับ 1.0", m_even, 1.0)
check("2.2 MAE ของชุดที่ 2 เท่ากับ 1.0", m_spiky, 1.0)
check("2.3 RMSE ชุดที่ 2 สูงกว่าชุดที่ 1 อย่างน้อย 1.9 เท่า", r_spiky / r_even >= 1.9, True)
check("2.4 เขียนคำอธิบายแล้ว", not answer_2.startswith("TODO"), True)

print("\nคำอธิบาย:", answer_2)

### 📝 ข้อ 2 กำลังสอนอะไร

นี่คือ **ตัวอย่างการคำนวณ 2.2** ในตำรา และเป็นเหตุผลเดียวที่ทำให้ต้องรายงานสองค่าเสมอ

| | ชุดที่ 1 (กระจายสม่ำเสมอ) | ชุดที่ 2 (พลาดหนักจุดเดียว) |
|---|---|---|
| ความผิดพลาด | 1, 1, 1, 1 | 0, 0, 0, 4 |
| MAE | (1+1+1+1)/4 = **1.0000** | (0+0+0+4)/4 = **1.0000** |
| RMSE | √(4/4) = **1.0000** | √(16/4) = **2.0000** |

**MAE มองไม่เห็นความต่างเลย** เพราะมันพิจารณาเพียงขนาดความผิดพลาดเฉลี่ยแบบเชิงเส้น
ความผิดพลาด 2 ดาวหนึ่งครั้ง ($2^2=4$) มีค่าเท่ากับความผิดพลาด 1 ดาวสี่ครั้งรวมกัน ($4\times1^2$)
**เฉพาะเมื่อวัดด้วย RMSE เท่านั้น** ที่สองกรณีนี้แยกออกจากกัน

**ทำไมเรื่องนี้สำคัญในงานจริง** ลองนึกถึงระบบแนะนำภาพยนตร์สองระบบ

* ระบบ ก ทายคลาดไป 1 ดาวกับทุกเรื่อง — ผู้ใช้รู้สึกว่า "พอใช้ได้ทั้งหมด"
* ระบบ ข ทายแม่นเป๊ะ 3 เรื่อง แต่แนะนำหนังที่ผู้ใช้เกลียดมากเป็นอันดับหนึ่ง

ทั้งสองได้ MAE เท่ากัน แต่ประสบการณ์ของผู้ใช้ต่างกันคนละโลก
**เลือกค่าหลักให้สอดคล้องกับต้นทุนของความผิดพลาดในงานจริง** — ถ้าการทายผิดรุนแรง
มีต้นทุนสูงเป็นพิเศษ ให้ใช้ RMSE เป็นตัวหลัก

---
## ข้อ 3 — Precision@K · Recall@K · F1@K ⭐
**หัวข้อ 2.2**

เขียนสามฟังก์ชันด้วยตัวเอง แล้วคำนวณกับ `RANK_A` และ `RANK_B` ที่ `K = 10`

$$P@K=\frac{|\mathcal{R}_u \cap \mathcal{L}^K_u|}{K} \qquad
R@K=\frac{|\mathcal{R}_u \cap \mathcal{L}^K_u|}{|\mathcal{R}_u|} \qquad
F1@K=\frac{2PR}{P+R}$$

**คำถามที่ต้องตอบให้ได้** ทั้งสองแบบจำลองวางรายการที่ถูกคนละตำแหน่ง (A: 1, 3, 5 / B: 4, 6, 9)
แต่ตัวชี้วัดสามตัวนี้ให้ค่าเท่ากันหรือไม่ **เพราะอะไร**

In [ ]:
# ============================== เฉลยข้อ 3 ==============================
def my_precision_at_k(ranked, relevant, k):
    top = list(ranked)[:k]
    return sum(1 for it in top if it in relevant) / k          # ตัวส่วนคือ K เสมอ


def my_recall_at_k(ranked, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return float("nan")
    top = list(ranked)[:k]
    return sum(1 for it in top if it in relevant) / len(relevant)   # ตัวส่วนคือ |R| ไม่ใช่ K


def my_f1_at_k(ranked, relevant, k):
    p = my_precision_at_k(ranked, relevant, k)
    r = my_recall_at_k(ranked, relevant, k)
    if np.isnan(r) or (p + r) == 0:
        return 0.0
    return 2 * p * r / (p + r)


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
pa = my_precision_at_k(RANK_A, RELEVANT, K)
pb = my_precision_at_k(RANK_B, RELEVANT, K)
ra = my_recall_at_k(RANK_A, RELEVANT, K)
f1a = my_f1_at_k(RANK_A, RELEVANT, K)

rows = []
for nm, rk in (("A", RANK_A), ("B", RANK_B)):
    rows.append({"Model": nm,
                 "P@3": my_precision_at_k(rk, RELEVANT, 3),
                 "P@10": my_precision_at_k(rk, RELEVANT, 10),
                 "R@10": my_recall_at_k(rk, RELEVANT, 10),
                 "F1@10": my_f1_at_k(rk, RELEVANT, 10),
                 "ตำแหน่งของถูก": [i + 1 for i, it in enumerate(rk) if it in RELEVANT]})
display(pd.DataFrame(rows).set_index("Model"))

check("3.1 Precision@10 ของ A", pa, 0.3000)
check("3.2 Precision@10 ของ B", pb, 0.3000)
check("3.3 Recall@10 ของ A", ra, 1.0000)
check("3.4 F1@10 ของ A", f1a, 0.4615)
check("3.5 Precision@3 ของ A", my_precision_at_k(RANK_A, RELEVANT, 3), 0.6667)
check("3.6 Precision@3 ของ B", my_precision_at_k(RANK_B, RELEVANT, 3), 0.0000)
check("3.7 F1@10 ของ A เท่ากับของ B", abs(f1a - my_f1_at_k(RANK_B, RELEVANT, K)) < 1e-9, True)

### 📝 ข้อ 3 กำลังสอนอะไร

**คำตอบของคำถามท้ายข้อ** ที่ `K = 10` ทั้งสามตัวชี้วัดให้ค่า **เท่ากันเป๊ะ**
เพราะทั้งสามตรวจเพียงว่ารายการ *"อยู่ภายใน Top-10 หรือไม่"* — ไม่สนใจว่าอยู่อันดับที่เท่าไร
Model A วางของถูกไว้ที่อันดับ 1, 3, 5 ส่วน Model B วางไว้ที่ 4, 6, 9 ซึ่งต่างกันมากในสายตาผู้ใช้
แต่ตัวชี้วัดกลุ่มนี้มองว่าเหมือนกันทุกประการ

**แต่ที่ `K = 3` แยกออกทันที** (A ได้ 0.6667 ส่วน B ได้ 0.0000) เพราะการลด K
ก็คือการบังคับให้ตัวชี้วัดสนใจเฉพาะส่วนต้นของลิสต์ — ซึ่งเป็นการ "จำลอง" ความไวต่อตำแหน่งแบบหยาบ ๆ

**จุดที่พลาดกันบ่อยที่สุด** ใช้ `K` เป็นตัวส่วนของ Recall
ตัวส่วนของ Recall คือ **จำนวนรายการที่ผู้ใช้ชอบจริงทั้งหมด** ($|\mathcal{R}_u| = 3$ ในข้อนี้)
ไม่ใช่ K ถ้าใช้ K จะได้ 0.3 แทนที่จะเป็น 1.0 และตัวเลขจะไม่มีความหมาย

**สมบัติเชิงโครงสร้างที่จริงเสมอ** (ไม่ใช่ความบังเอิญของข้อมูลชุดนี้)

* `Recall@K` **ไม่มีวันลดลง** เมื่อ K เพิ่ม — ตัวเศษเพิ่มได้ ตัวส่วนคงที่
* `Precision@K` **มีแนวโน้มลดลง** เมื่อ K เพิ่ม — ตัวส่วนโตเป็นเชิงเส้น
* จึง **ห้ามเปรียบเทียบค่าที่ K ต่างกันโดยเด็ดขาด**

**เรื่อง Precision@10 = 0.30 ที่ต้องระวังการตีความ** ค่านี้หมายถึง
"3 จาก 10 รายการตรงกับความเกี่ยวข้องที่สังเกตได้" **ไม่ใช่** ข้อสรุปว่าอีก 7 รายการไม่มีประโยชน์
รายการที่ไม่อยู่ใน Ground Truth อาจเป็นรายการที่ผู้ใช้ยังไม่เคยเห็น (Pseudo False Positive)

---
## ข้อ 4 — Reciprocal Rank และ MRR@K ⭐⭐
**หัวข้อ 2.3.1**

$$RR@K=\frac{1}{\text{อันดับของรายการเกี่ยวข้องชิ้นแรก}} \qquad
MRR@K=\frac{1}{|\mathcal{U}|}\sum_{u} RR@K(u)$$

ถ้าไม่พบรายการเกี่ยวข้องเลยใน Top-K ให้ `RR = 0`

ใช้ข้อมูล `THREE_USERS` (ผู้ใช้ 3 คน แต่ละคนมีลิสต์ของ Model A และ Model B)

| ผู้ใช้ | Model A วางของถูกที่อันดับ | Model B วางของถูกที่อันดับ |
|---|---|---|
| Emma | 1, 3, 5 | 4, 6, 9 |
| Liam | 2, 4, 7 | 1, 5, 8 |
| Noah | 2, 5, 6 | 4, 7, 10 |

**คำถามที่ต้องตอบให้ได้** RR มีจุดอ่อนสำคัญอะไร (ลองคิดถึงรายการที่ถูกชิ้นที่สองเป็นต้นไป)

In [ ]:
# ============================== เฉลยข้อ 4 ==============================
def my_reciprocal_rank(ranked, relevant, k):
    """ส่วนกลับของอันดับรายการเกี่ยวข้องชิ้นแรกใน Top-K  (ไม่พบเลย = 0)"""
    seq = list(ranked)[:k]                 # ต้องตัดที่ K *ก่อน* ค้นหา
    for pos, it in enumerate(seq, start=1):
        if it in relevant:
            return 1.0 / pos               # เจอชิ้นแรกแล้วคืนค่าทันที
    return 0.0


def my_mrr_at_k(model_key, users_data, k):
    return float(np.mean([my_reciprocal_rank(d[model_key], d["relevant"], k)
                          for d in users_data.values()]))


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
print(f"{'ผู้ใช้':<8}{'RR ของ A':>12}{'RR ของ B':>12}{'อันดับแรกที่เจอ (A / B)':>28}")
print("-" * 62)
for u, d in THREE_USERS.items():
    fa = next(i + 1 for i, it in enumerate(d["A"]) if it in d["relevant"])
    fb = next(i + 1 for i, it in enumerate(d["B"]) if it in d["relevant"])
    print(f"{u:<8}{my_reciprocal_rank(d['A'], d['relevant'], K):>12.4f}"
          f"{my_reciprocal_rank(d['B'], d['relevant'], K):>12.4f}{f'{fa} / {fb}':>28}")
print()

check("4.1 RR ของ Emma กับ Model A",
      my_reciprocal_rank(THREE_USERS["Emma"]["A"], THREE_USERS["Emma"]["relevant"], K), 1.0000)
check("4.2 RR ของ Emma กับ Model B",
      my_reciprocal_rank(THREE_USERS["Emma"]["B"], THREE_USERS["Emma"]["relevant"], K), 0.2500)
check("4.3 MRR@10 ของ Model A", my_mrr_at_k("A", THREE_USERS, K), 0.6667)
check("4.4 MRR@10 ของ Model B", my_mrr_at_k("B", THREE_USERS, K), 0.5000)
check("4.5 ไม่พบรายการเกี่ยวข้องเลย -> RR = 0",
      my_reciprocal_rank(["n1", "n2", "n3"], {"r1"}, 3), 0.0000)
check("4.6 พบนอก Top-K -> RR = 0 (ตัดที่ K ก่อน)",
      my_reciprocal_rank(["n1", "n2", "r3"], {"r3"}, 2), 0.0000)

print(f"\nMRR@10 : A = {my_mrr_at_k('A', THREE_USERS, K):.4f}  "
      f"B = {my_mrr_at_k('B', THREE_USERS, K):.4f}  "
      f"-> A สูงกว่า {my_mrr_at_k('A', THREE_USERS, K) - my_mrr_at_k('B', THREE_USERS, K):.4f} จุด")

### 📝 ข้อ 4 กำลังสอนอะไร

**คำตอบของคำถามท้ายข้อ — จุดอ่อนของ RR**
RR สนใจ **เฉพาะรายการที่ถูกชิ้นแรก** เท่านั้น หลังพบแล้วคะแนนไม่เปลี่ยนอีกเลย

ลองพิจารณาสองลิสต์นี้ (ผู้ใช้ชอบ 3 รายการ)

| | ตำแหน่งของถูก | RR | สิ่งที่ผู้ใช้รู้สึก |
|---|---|---|---|
| ลิสต์ ก | 1 เท่านั้น (อีกสองชิ้นหลุด Top-10) | 1.0000 | เจอของดีชิ้นเดียว |
| ลิสต์ ข | 1, 2, 3 | 1.0000 | เจอของดีสามชิ้นติดกัน |

**RR ให้คะแนนเท่ากันเป๊ะ** ทั้งที่ลิสต์ ข ดีกว่าอย่างชัดเจน
ถ้างานต้องการหลายรายการในลิสต์มีคุณภาพ ต้องใช้ `MAP@K` หรือ `nDCG@K` ร่วมด้วย

**RR เหมาะกับงานแบบใด** งานที่ผู้ใช้ต้องการ *คำตอบเดียวที่ถูก* เช่น
การค้นเอกสารในองค์กร คำถาม-คำตอบ หรือการค้นหาสินค้าที่ระบุชื่อมาแล้ว

**จุดที่พลาดกันบ่อยที่สุด** ลืมตัดลิสต์ที่ K ก่อนค้นหา
ถ้าของถูกอยู่อันดับ 15 แต่ K = 10 คำตอบต้องเป็น 0 ไม่ใช่ 1/15
(ข้อ 4.6 ในเซลล์ตรวจคำตอบดักกรณีนี้ไว้โดยเฉพาะ)

**เหตุใด RR แยกความต่างได้แรงที่สุด** ในกรณีของ Emma RR ของ A สูงกว่า B ถึง **4 เท่า**
(1.0000 เทียบกับ 0.2500) ขณะที่ nDCG ต่างกันเพียง 1.73 เท่า
เพราะ $1/\rho$ ลดลงเร็วกว่า $1/\log_2(\rho+1)$ มาก
→ **ขนาดของความต่างขึ้นกับตัวชี้วัดที่เลือก ไม่ใช่แค่ว่าใครชนะ**

---
## ข้อ 5 — Average Precision และ MAP@K ⭐⭐
**หัวข้อ 2.3.2**

$$AP@K=\frac{1}{\min(K,|\mathcal{R}_u|)}\sum_{p=1}^{K} \text{rel}(p)\cdot P@p$$

คือ เฉลี่ยค่า Precision ณ **ทุกตำแหน่งที่พบรายการเกี่ยวข้อง**

**จุดที่พลาดกันบ่อยที่สุด** ตัวส่วนคือ $\min(K, |\mathcal{R}_u|)$ ไม่ใช่ $K$ และไม่ใช่จำนวนที่หาเจอ
เหตุผล: ถ้าผู้ใช้ชอบ 3 รายการ ก็เก็บได้สูงสุด 3 ชิ้น การหารด้วย 10 จะทำให้คะแนนต่ำเกินจริงเสมอ

**สิ่งที่ต้องทำเพิ่ม** เขียนฟังก์ชัน `ap_trace` ที่คืน *ลิสต์ของค่า Precision ณ จุดที่พบของถูก*
เพื่อให้ตรวจการคำนวณด้วยมือได้

In [ ]:
# ============================== เฉลยข้อ 5 ==============================
def ap_trace(ranked, relevant, k):
    """คืนลิสต์ของ Precision ณ ทุกตำแหน่งที่พบรายการเกี่ยวข้อง (ใน Top-K)"""
    seq = list(ranked)[:k]
    hits, terms = 0, []
    for pos, it in enumerate(seq, start=1):
        if it in relevant:
            hits += 1
            terms.append(hits / pos)        # = Precision@pos ณ จุดที่เจอ
    return terms


def my_average_precision_at_k(ranked, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return float("nan")
    terms = ap_trace(ranked, relevant, k)
    return sum(terms) / min(len(relevant), k)     # ตัวส่วนคือ min(K, |R|)


def my_map_at_k(model_key, users_data, k):
    return float(np.mean([my_average_precision_at_k(d[model_key], d["relevant"], k)
                          for d in users_data.values()]))


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
ta = ap_trace(RANK_A, RELEVANT, K)
tb = ap_trace(RANK_B, RELEVANT, K)
print(f"Model A พบของถูกที่อันดับ 1, 3, 5 -> Precision = "
      f"{' , '.join(f'{n+1}/{p}' for n, p in zip(range(3), [1, 3, 5]))} = {[round(x,4) for x in ta]}")
print(f"        AP@10 = ({' + '.join(f'{x:.4f}' for x in ta)}) / 3 = {sum(ta)/3:.4f}")
print(f"Model B พบของถูกที่อันดับ 4, 6, 9 -> {[round(x,4) for x in tb]}")
print(f"        AP@10 = ({' + '.join(f'{x:.4f}' for x in tb)}) / 3 = {sum(tb)/3:.4f}\n")

check("5.1 ap_trace ของ A", ta, [1.0, 2/3, 0.6])
check("5.2 ap_trace ของ B", tb, [0.25, 1/3, 1/3])
check("5.3 AP@10 ของ A", my_average_precision_at_k(RANK_A, RELEVANT, K), 0.7556)
check("5.4 AP@10 ของ B", my_average_precision_at_k(RANK_B, RELEVANT, K), 0.3056)
check("5.5 MAP@10 ของ Model A", my_map_at_k("A", THREE_USERS, K), 0.5661)
check("5.6 MAP@10 ของ Model B", my_map_at_k("B", THREE_USERS, K), 0.3919)
check("5.7 ตัวส่วนใช้ min(K, |R|) ไม่ใช่ K",
      my_average_precision_at_k(["r1"], {"r1"}, 10), 1.0000)

_apa = my_average_precision_at_k(RANK_A, RELEVANT, K)
_apb = my_average_precision_at_k(RANK_B, RELEVANT, K)
print(f"\nAP@10 ของ A สูงกว่า B {_apa/_apb:.2f} เท่า ทั้งที่ Precision@10 เท่ากันเป๊ะ")

### 📝 ข้อ 5 กำลังสอนอะไร

**AP ต่างจาก RR อย่างไร** RR ดูเฉพาะของถูก *ชิ้นแรก* ส่วน AP ใช้ข้อมูลจากของถูก **ทุกชิ้น** ใน Top-K
แล้วให้น้ำหนักชิ้นที่อยู่อันดับต้นมากกว่าโดยอัตโนมัติ เพราะ Precision ณ ตำแหน่งต้น ๆ สูงกว่าเสมอ

การคำนวณของ Model A ทีละพจน์

| พบของถูกที่อันดับ | ของถูกสะสม | Precision ณ จุดนั้น |
|---|---|---|
| 1 | 1 | 1/1 = 1.0000 |
| 3 | 2 | 2/3 = 0.6667 |
| 5 | 3 | 3/5 = 0.6000 |

$$AP@10 = \frac{1.0000+0.6667+0.6000}{\min(10,\,3)} = \frac{2.2667}{3} = 0.7556$$

**จุดที่พลาดกันบ่อยที่สุด — ตัวส่วน**

* หารด้วย `K` (=10) → ได้ 0.2267 ซึ่งต่ำกว่าความจริง 3.3 เท่า และไม่มีวันเข้าใกล้ 1.0 ได้เลย
* หารด้วย "จำนวนที่หาเจอ" → ได้ 0.7556 เท่ากันในกรณีนี้ (เพราะเจอครบ 3) แต่จะผิดทันที
  เมื่อของถูกบางชิ้นหลุด Top-K เพราะระบบที่หาเจอชิ้นเดียวแต่วางไว้อันดับ 1 จะได้ AP = 1.0 ทันที

ตัวส่วนที่ถูกคือ $\min(K, |\mathcal{R}_u|)$ — **"เก็บได้มากที่สุดกี่ชิ้น"**
ถ้าผู้ใช้ชอบ 3 รายการ ก็เก็บได้สูงสุด 3 ชิ้น ไม่ว่า K จะใหญ่แค่ไหน

**MAP คืออะไร** คือค่าเฉลี่ยของ AP ข้ามผู้ใช้ทุกคน (macro-average ระดับผู้ใช้)
ต้องระวังว่าผู้ใช้ที่มีรายการเกี่ยวข้องน้อยจะมีน้ำหนักเท่ากับผู้ใช้ที่มีมาก — ซึ่งเป็นสิ่งที่ต้องการ
ในการประเมินระบบผู้แนะนำ เพราะหน่วยของการวัดคือ "ผู้ใช้" ไม่ใช่ "เหตุการณ์"

---
## ข้อ 6 — DCG · IDCG · nDCG@K ⭐⭐⭐
**หัวข้อ 2.3.3**

$$DCG@K=\sum_{p=1}^{K}\frac{g(\text{rel}_p)}{\log_2(p+1)} \qquad
nDCG@K=\frac{DCG@K}{IDCG@K}$$

เขียนสามฟังก์ชัน

1. `my_dcg(gains, k, gain_type)` — รองรับทั้ง `"linear"` ($g=\text{rel}$) และ `"exponential"` ($g=2^{\text{rel}}-1$)
2. `my_idcg(relevance, k, gain_type)` — DCG ของลิสต์อุดมคติ (เรียง rel จากมากไปน้อย)
3. `my_ndcg_at_k(ranked, relevance, k, gain_type)`

**ข้อควรระวังสองข้อ**

* IDCG ขึ้นกับ **ผู้ใช้** ไม่ใช่ **แบบจำลอง** — A กับ B จึงหารด้วยค่าเดียวกัน
* ต้องประกาศ `gain_type` ทุกครั้งที่รายงาน เพราะสองแบบให้ค่าต่างกันมากเมื่อ rel > 1

In [ ]:
# ============================== เฉลยข้อ 6 ==============================
def my_dcg(gains, k, gain_type="linear"):
    g = np.asarray(gains, dtype=float)[:k]
    if gain_type == "exponential":
        g = np.power(2.0, g) - 1.0
    elif gain_type != "linear":
        raise ValueError("gain_type ต้องเป็น 'linear' หรือ 'exponential'")
    discounts = 1.0 / np.log2(np.arange(2, len(g) + 2))     # 1/log2(p+1) เมื่อ p เริ่มที่ 1
    return float(np.sum(g * discounts))


def my_idcg(relevance, k, gain_type="linear"):
    """ลิสต์อุดมคติ = เรียง rel จากมากไปน้อย  ขึ้นกับ *ผู้ใช้* ไม่ใช่แบบจำลอง"""
    ideal = sorted(relevance.values(), reverse=True)
    return my_dcg(ideal, k, gain_type)


def my_ndcg_at_k(ranked, relevance, k, gain_type="linear"):
    gains = [relevance.get(it, 0.0) for it in list(ranked)[:k]]   # ไม่อยู่ใน dict = rel 0
    dcg = my_dcg(gains, k, gain_type)
    idcg = my_idcg(relevance, k, gain_type)
    return 0.0 if idcg == 0 else dcg / idcg


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
rel_map = {i: 1.0 for i in RELEVANT}

print("การคำนวณทีละตำแหน่งของ Model A (ของถูกอยู่อันดับ 1, 3, 5)")
print(f"  DCG@10 = 1/log2(2) + 1/log2(4) + 1/log2(6)")
print(f"         = {1/np.log2(2):.4f} + {1/np.log2(4):.4f} + {1/np.log2(6):.4f}"
      f" = {my_ndcg_at_k(RANK_A, rel_map, K)*my_idcg(rel_map, K):.4f}")
print(f"  IDCG@10 (ของถูกอยู่อันดับ 1, 2, 3) = {1/np.log2(2):.4f} + {1/np.log2(3):.4f}"
      f" + {1/np.log2(4):.4f} = {my_idcg(rel_map, K):.4f}")
print(f"  nDCG@10 = {my_ndcg_at_k(RANK_A, rel_map, K)*my_idcg(rel_map, K):.4f} /"
      f" {my_idcg(rel_map, K):.4f} = {my_ndcg_at_k(RANK_A, rel_map, K):.4f}\n")

check("6.1 DCG@10 ของ A (rel ทวิภาค)", my_dcg([1, 0, 1, 0, 1, 0, 0, 0, 0, 0], K), 1.8869)
check("6.2 IDCG@10 (ของถูก 3 ชิ้นอยู่อันดับ 1-3)", my_idcg(rel_map, K), 2.1309)
check("6.3 nDCG@10 ของ A", my_ndcg_at_k(RANK_A, rel_map, K), 0.8855)
check("6.4 nDCG@10 ของ B", my_ndcg_at_k(RANK_B, rel_map, K), 0.5105)
check("6.5 IDCG เท่ากันทั้งสองแบบจำลอง", my_idcg(rel_map, K), my_idcg(rel_map, K))
check("6.6 จัดอันดับสมบูรณ์แบบ -> nDCG = 1.0",
      my_ndcg_at_k(["i2", "i5", "i9", "i1", "i3"], rel_map, K), 1.0000)
check("6.7 gain แบบ exponential กับ rel หลายระดับ",
      my_dcg([3, 2, 0, 1], 4, "exponential"), 9.3235)
check("6.8 IDCG = 0 -> คืน 0.0", my_ndcg_at_k(RANK_A, {}, K), 0.0000)

r = my_ndcg_at_k(RANK_A, rel_map, K) / my_ndcg_at_k(RANK_B, rel_map, K)
print(f"\nnDCG@10 ของ A สูงกว่า B {r:.2f} เท่า (เทียบกับ MRR ที่ต่างกัน 4.00 เท่า)")
print("เปรียบเทียบ gain สองแบบบน rel = [3, 2, 0, 1]")
print(f"  linear      = {my_dcg([3,2,0,1], 4, 'linear'):.4f}")
print(f"  exponential = {my_dcg([3,2,0,1], 4, 'exponential'):.4f}   ต่างกัน "
      f"{my_dcg([3,2,0,1],4,'exponential')/my_dcg([3,2,0,1],4,'linear'):.2f} เท่า")

### 📝 ข้อ 6 กำลังสอนอะไร

**สามขั้นของ nDCG**

1. **DCG จากลิสต์จริง** — ผลได้หารด้วยส่วนลดตามตำแหน่ง $1/\log_2(p+1)$
2. **IDCG จากลิสต์อุดมคติ** — เรียงความเกี่ยวข้องจากมากไปน้อย
3. **หาร** → ได้ค่าในช่วง $[0,1]$ เสมอ จึงเทียบข้ามผู้ใช้ที่มีจำนวนรายการชอบไม่เท่ากันได้

**จุดที่พลาดกันบ่อยที่สุด — สร้าง IDCG ผิด**
IDCG ขึ้นกับ **ผู้ใช้** เท่านั้น (ว่าเขาชอบกี่รายการ และแต่ละรายการ rel เท่าใด)
**ไม่ขึ้นกับผลลัพธ์ของแบบจำลอง** ดังนั้น Model A และ Model B ต้องหารด้วยค่าเดียวกัน = 2.1309
ถ้าเขียนโค้ดโดยเรียง `gains` ของแบบจำลองเองแทนที่จะเรียง `relevance.values()`
จะได้ IDCG ของ B เป็น 1.8869 ซึ่งทำให้ nDCG ของ B พุ่งไปเกือบ 0.58 — สูงกว่าความจริง

**ทำไมส่วนลดต้องเป็น $\log_2(p+1)$** เพราะต้องการให้ตำแหน่ง 1 ได้ส่วนลด = 1 พอดี
($\log_2 2 = 1$) และให้คะแนนลดลงช้ากว่า $1/p$ ของ RR สะท้อนว่าผู้ใช้ยังมีโอกาสเห็นอันดับ 5–10 อยู่บ้าง

**เรื่อง `gain_type` ที่ต้องประกาศเสมอ** บน rel = [3, 2, 0, 1]

| gain_type | สูตร | DCG |
|---|---|---|
| linear | $g = \text{rel}$ | 4.6479 |
| exponential | $g = 2^{\text{rel}}-1$ | 9.3235 |

ต่างกัน **2 เท่า** จากข้อมูลชุดเดียวกันเป๊ะ การรายงานว่า "nDCG@10 = 0.72" โดยไม่บอก gain function
จึงเป็นตัวเลขที่ทำซ้ำไม่ได้ (เมื่อความเกี่ยวข้องเป็นทวิภาค 0/1 ทั้งสองแบบให้ค่าเท่ากันพอดี
เพราะ $2^1-1=1$ และ $2^0-1=0$ — จึงไม่ค่อยมีใครสังเกตปัญหานี้จนกว่าจะใช้ rel หลายระดับ)

---
## ข้อ 7 — AUC ด้วยการนับคู่ ⭐⭐
**หัวข้อ 2.4**

$$AUC=\frac{\text{จำนวนคู่ (เกี่ยวข้อง, ไม่เกี่ยวข้อง) ที่เรียงถูก}}{|\mathcal{R}_u|\times|\mathcal{N}_u|}$$

**ตีความ** ความน่าจะเป็นที่รายการเกี่ยวข้องซึ่งสุ่มมาหนึ่งชิ้น จะถูกจัดอันดับ *สูงกว่า*
รายการไม่เกี่ยวข้องซึ่งสุ่มมาหนึ่งชิ้น — ไม่ใช่ "ความแม่นยำ"

ในลิสต์ของเรามีของถูก 3 ชิ้น และของไม่ถูก 7 ชิ้น จึงมีคู่ทั้งหมด $3\times 7=21$ คู่

เขียน `my_auc(ranked, relevant)` และ `count_correct_pairs(ranked, relevant)`
แล้วตอบคำถาม: **AUC ของ Model B ต่ำกว่า 0.5 แปลว่าอะไร**

In [ ]:
# ============================== เฉลยข้อ 7 ==============================
def count_correct_pairs(ranked, relevant):
    ranked = list(ranked)
    rel_pos = [i for i, it in enumerate(ranked) if it in relevant]
    irr_pos = [i for i, it in enumerate(ranked) if it not in relevant]
    return sum(1 for rp in rel_pos for ip in irr_pos if rp < ip)


def my_auc(ranked, relevant):
    ranked = list(ranked)
    n_rel = sum(1 for it in ranked if it in relevant)
    n_irr = len(ranked) - n_rel
    if n_rel == 0 or n_irr == 0:
        return float("nan")
    return count_correct_pairs(ranked, relevant) / (n_rel * n_irr)


# --------------------------- ตรวจคำตอบ ---------------------------
for nm, rk in (("A", RANK_A), ("B", RANK_B)):
    pos_rel = [i + 1 for i, it in enumerate(rk) if it in RELEVANT]
    good = count_correct_pairs(rk, RELEVANT)
    print(f"Model {nm}: ของถูกอยู่อันดับ {pos_rel}  ->  คู่ที่เรียงถูก {good} จาก 21 คู่"
          f"  ->  AUC = {good}/21 = {my_auc(rk, RELEVANT):.4f}")
print()

check("7.1 จำนวนคู่ที่เรียงถูกของ A", count_correct_pairs(RANK_A, RELEVANT), 18)
check("7.2 จำนวนคู่ที่เรียงถูกของ B", count_correct_pairs(RANK_B, RELEVANT), 8)
check("7.3 AUC ของ A", my_auc(RANK_A, RELEVANT), 0.8571)
check("7.4 AUC ของ B", my_auc(RANK_B, RELEVANT), 0.3810)
check("7.5 เรียงถูกทุกคู่ -> AUC = 1.0",
      my_auc(["i2", "i5", "i9", "i1", "i3", "i4"], RELEVANT), 1.0000)
check("7.6 เรียงกลับด้านทั้งหมด -> AUC = 0.0",
      my_auc(["i1", "i3", "i4", "i2", "i5", "i9"], RELEVANT), 0.0000)

### 📝 ข้อ 7 กำลังสอนอะไร

**คำตอบของคำถามท้ายข้อ — AUC ของ B ต่ำกว่า 0.5 แปลว่าอะไร**
AUC = 0.5 คือการจัดอันดับแบบ **สุ่มล้วน** ดังนั้น AUC = 0.3810 แปลว่า Model B
จัดอันดับ **แย่กว่าการเดาสุ่มอย่างเป็นระบบ** — ถ้าสลับลิสต์กลับหัวจะได้ผลดีกว่าเสียอีก
(สังเกตว่า $0.3810 = 1 - 0.6190$ และ AUC ของลิสต์กลับด้านคือ $1 - \text{AUC}$ เสมอ)

**การตีความที่ถูกต้อง** AUC = 0.8571 หมายถึง
*"ถ้าสุ่มรายการเกี่ยวข้องมาหนึ่งชิ้น และรายการไม่เกี่ยวข้องมาหนึ่งชิ้น มีโอกาส 85.71%
ที่ระบบจะจัดรายการเกี่ยวข้องไว้สูงกว่า"*

**ไม่ใช่** "ความแม่นยำ 85.71%" — นี่คือความเข้าใจผิดที่พบบ่อยที่สุดเกี่ยวกับ AUC
เพราะ AUC วัด *คุณภาพการเรียงลำดับคู่* ไม่ได้วัดว่าทำนายถูกกี่เปอร์เซ็นต์

**ข้อจำกัดสำคัญ — AUC ให้น้ำหนักทุกคู่เท่ากันหมด**
การที่รายการเกี่ยวข้องเลื่อนจากอันดับ 400 ขึ้นมา 399 (แซงของไม่เกี่ยวข้องหนึ่งชิ้น)
ทำให้ AUC เพิ่มขึ้น **เท่ากับ** การเลื่อนจากอันดับ 4 ขึ้นมา 3

แต่ในสายตาผู้ใช้ การเลื่อนจาก 4 เป็น 3 มีผลมหาศาล ส่วนการเปลี่ยนแปลงที่อันดับ 399–400
อาจไม่มีผลต่อผู้ใช้เลยเพราะไม่มีใครเลื่อนไปถึง

> **ข้อปฏิบัติ** AUC เหมาะกับการประเมิน *ความสามารถในการจัดอันดับโดยรวม*
> แต่ถ้าระบบแสดงผลเพียง 10 อันดับแรก ให้ใช้ `nDCG@10`, `Precision@10` หรือ `Recall@10`
> เป็น Primary Metric แทน และต้อง **ประกาศ Candidate Set ให้ชัด** เพราะการเปลี่ยนรายการ
> ที่นำมาเปรียบเทียบเปลี่ยนค่า AUC ได้

---
## ข้อ 8 — Catalog Coverage และ Gini ของการเปิดเผย ⭐⭐
**หัวข้อ 2.5.1–2.5.2**

ใช้ `MODEL_X` (ยอดนิยมชุดเดียวให้ทุกคน) และ `MODEL_Y` (เฉพาะบุคคล) บนคลัง 12 เรื่อง
ผู้ใช้ 8 คน K = 5 → ช่องแสดงผลทั้งระบบ $8\times 5 = 40$ ช่อง

$$\text{Coverage@K}=\frac{|\bigcup_u \mathcal{L}^K_u|}{|\mathcal{I}|} \qquad
G=\frac{\sum_i (2i-n-1)\,x_i}{n\sum_i x_i}$$

**กฎบังคับสองข้อของ Gini ที่พลาดกันบ่อยที่สุด**

1. ต้อง **เรียง $x$ จากน้อยไปมากก่อน** เสมอ — ลืมเรียงแล้วสูตรคืนค่าที่ไม่มีความหมาย
2. รายการที่ **ไม่เคยถูกแนะนำเลยต้องอยู่ในเวกเตอร์ในฐานะ 0** — ถ้าตัดทิ้ง
   ค่าจะต่ำกว่าความจริง เพราะกลุ่มที่ถูกกีดกันที่สุดหายไปจากการคำนวณพอดี

**ตรวจทานง่าย ๆ** $\sum_i (2i-n-1)=0$ เสมอ และ $\sum_i x_i$ ต้องเท่ากับ 40 พอดี

In [ ]:
# ============================== เฉลยข้อ 8 ==============================
def exposure_vector(model_dict, all_items, users):
    cnt = Counter()
    for u in users:
        cnt.update(model_dict[u])
    return [cnt.get(i, 0) for i in all_items]      # รายการที่ไม่เคยถูกแนะนำ = 0 ต้องอยู่ในลิสต์


def my_coverage(model_dict, n_items, users):
    shown = set()
    for u in users:
        shown.update(model_dict[u])
    return len(shown) / n_items


def my_gini(values):
    v = np.array(sorted(values), dtype=float)      # ขั้นบังคับ: เรียงจากน้อยไปมากก่อน
    n = len(v)
    if v.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * v).sum() / (n * v.sum()))


# --------------------------- ตรวจคำตอบ ---------------------------
ex_x = exposure_vector(MODEL_X, IDS, USERS8)
ex_y = exposure_vector(MODEL_Y, IDS, USERS8)
print("การเปิดเผยของ Model X :", ex_x, " รวม", sum(ex_x))
print("การเปิดเผยของ Model Y :", ex_y, " รวม", sum(ex_y))

# ตารางตรวจทีละพจน์
for nm, ex in (("X", ex_x), ("Y", ex_y)):
    v = np.array(sorted(ex), dtype=float)
    n = len(v)
    w = 2 * np.arange(1, n + 1) - n - 1
    print(f"\nModel {nm}")
    print(f"  xᵢ เรียงแล้ว : {[int(x) for x in v]}")
    print(f"  2i − n − 1   : {[int(x) for x in w]}   (Σ = {int(w.sum())} ✓ ต้องเป็น 0 เสมอ)")
    print(f"  ผลคูณ        : {[int(x) for x in v*w]}  ->  Σ = {int((v*w).sum())}")
    print(f"  Gini = {int((v*w).sum())} / (12 × {int(v.sum())}) = {my_gini(ex):.4f}")
print()

check("8.1 ผลรวมการเปิดเผยของ X = 40", sum(ex_x), 40)
check("8.2 ผลรวมการเปิดเผยของ Y = 40", sum(ex_y), 40)
check("8.3 ความยาวเวกเตอร์ = 12 (รวมรายการที่ได้ 0)", len(ex_x), 12)
check("8.4 Coverage@5 ของ Model X", my_coverage(MODEL_X, 12, USERS8), 0.4167)
check("8.5 Coverage@5 ของ Model Y", my_coverage(MODEL_Y, 12, USERS8), 1.0000)
check("8.6 Gini ของ Model X", my_gini(ex_x), 0.5833)
check("8.7 Gini ของ Model Y", my_gini(ex_y), 0.1500)
check("8.8 กระจายเท่ากันหมด -> Gini = 0", my_gini([5, 5, 5, 5]), 0.0000)
check("8.9 ต้องเรียงก่อนคำนวณ (สลับลำดับต้องได้ค่าเดิม)",
      my_gini([0, 8, 0, 3]), my_gini([8, 3, 0, 0]))

# สาธิตกับดัก: ตัดรายการที่ได้ 0 ทิ้ง
nz = [c for c in ex_x if c > 0]
print(f"\n⚠ กับดัก: ถ้าตัดรายการที่ได้ 0 ทิ้ง Gini ของ Model X จะกลายเป็น {my_gini(nz):.4f}")
print(f"   ซึ่งต่ำกว่าความจริง ({my_gini(ex_x):.4f}) อย่างสิ้นเชิง")
print("   เพราะกลุ่มที่ถูกกีดกันมากที่สุด คือกลุ่มที่หายไปจากการคำนวณพอดี")

### 📝 ข้อ 8 กำลังสอนอะไร

**ทำไมต้องมีทั้ง Coverage และ Gini**

* **Coverage** ตอบว่า "มีรายการกี่ชิ้นที่ *เคย* ถูกแนะนำอย่างน้อยหนึ่งครั้ง"
* **Gini** ตอบว่า "การเปิดเผยนั้น *กระจายอย่างเท่าเทียม* เพียงใด"

ระบบที่แนะนำรายการหนึ่งชิ้นเป็นพันครั้ง และอีกสี่สิบเก้าชิ้นชิ้นละครั้ง
มี Coverage **เท่ากับ** ระบบที่แนะนำทั้งห้าสิบชิ้นเท่า ๆ กัน — Gini เท่านั้นที่จับความต่างนี้ได้

**กฎบังคับข้อ 1 — ต้องเรียงก่อน**
สูตร $\sum(2i-n-1)x_i$ อาศัยสมมติฐานว่า $x$ เรียงจากน้อยไปมาก
ถ้าลืมเรียง สัมประสิทธิ์จะไปจับคู่กับค่าผิดตัว และผลลัพธ์ไม่มีความหมายทางคณิตศาสตร์เลย
(ข้อ 8.9 ในเซลล์ตรวจคำตอบดักกรณีนี้: สลับลำดับอินพุตแล้วต้องได้ค่าเดิม)

**กฎบังคับข้อ 2 — รายการที่ได้ศูนย์ต้องอยู่ในเวกเตอร์**
นี่คือ **"กับดัก Counter"** ที่ตำราเตือนไว้ ถ้าเขียน

```python
cnt = Counter(); [cnt.update(l) for l in lists]
gini(list(cnt.values()))        # ❌ ผิด — รายการที่ไม่เคยถูกแนะนำหายไปหมด
```

Model X แนะนำเพียง 5 เรื่อง เรื่องละ 8 ครั้งเท่ากันหมด → ถ้าตัดอีก 7 เรื่องที่ได้ 0 ทิ้ง
เวกเตอร์จะกลายเป็น [8,8,8,8,8] ซึ่ง **Gini = 0.0000 "เท่าเทียมสมบูรณ์"**
ทั้งที่ความจริงคือ 0.5833 — กลับด้านจากความเป็นจริงโดยสิ้นเชิง

**ตรวจทานสองข้อที่ควรทำทุกครั้ง**

1. $\sum(2i-n-1) = 0$ เสมอ (เพราะสมมาตรรอบจุดกึ่งกลาง)
2. $\sum x_i$ ต้องเท่ากับจำนวนช่องแสดงผลทั้งระบบ = $8 \times 5 = 40$ พอดี

**ข้อควรระวังเรื่องการตีความ** Coverage@5 ของ Model Y = 1.0000 ไม่ได้แปลว่า Model Y ดีกว่า
มันบอกเพียงว่าคลังถูกใช้ประโยชน์กว้างกว่า — ต้องอ่านคู่กับ Precision/nDCG เสมอ
ไม่เช่นนั้น **ระบบสุ่มจะชนะทุกครั้ง** (ดูส่วนที่ 8 ของ `Lab2_All_in_One.ipynb` ที่ Random
ได้ Coverage 100% และ Gini ต่ำสุด แต่ nDCG แย่ที่สุด)

---
## ข้อ 9 — Novelty และ LongTailRatio ⭐⭐
**หัวข้อ 2.5.3–2.5.4**

$$\text{Novelty}@K=\frac{1}{K}\sum_{i\in\mathcal{L}^K_u}-\log_2 p(i),
\qquad p(i)=\frac{\text{ผู้เคยชม } i}{\text{ฐานผู้ใช้ทั้งแพลตฟอร์ม}}$$

ฐานผู้ใช้ทั้งแพลตฟอร์ม = `PLATFORM_USERS` = 500 คน · จำนวนผู้เคยชมอยู่ในคอลัมน์ `viewers`

**เส้นแบ่งหัว–หาง** ใช้เกณฑ์ *หนึ่งในสาม*: เรียงคลังตามความนิยมจากมากไปน้อย
แล้วไล่สะสมจนถึง 1/3 ของปฏิสัมพันธ์รวม รายการก่อนจุดนั้น = หัว ที่เหลือ = หาง

$$\text{LongTailRatio}@K=\frac{|\mathcal{L}^K_u \cap \text{Tail}|}{K}$$

**โจทย์เสริม** ข้อสุดท้ายให้ลอง "ปั่น" ค่า Novelty ของ Model X ให้สูงขึ้นอย่างน้อย 40%
โดย **เปลี่ยนรายการเพียงหนึ่งชิ้น** — เพื่อให้เห็นช่องโหว่ของตัวชี้วัดนี้ด้วยตัวเอง

In [ ]:
# ============================== เฉลยข้อ 9 ==============================
def my_novelty(reclist, viewers, n_platform_users):
    vals = []
    for it in reclist:
        p = viewers.get(it, 0) / n_platform_users
        p = max(p, 1.0 / (n_platform_users * 10))       # กันกรณี log(0)
        vals.append(-np.log2(p))                        # ต้องเป็น log ฐาน 2 (หน่วย "บิต")
    return float(np.mean(vals)) if vals else 0.0


def my_head_tail_split(viewers, head_share=0.33):
    items = sorted(viewers.items(), key=lambda kv: -kv[1])   # เรียงความนิยมมาก -> น้อย
    total = sum(viewers.values())
    head, acc = set(), 0
    for it, c in items:
        if acc >= head_share * total:                        # ตรวจ *ก่อน* บวก
            break
        head.add(it)
        acc += c
    return head, set(viewers) - head


def my_long_tail_ratio(reclist, tail_items):
    lst = list(reclist)
    if not lst:
        return 0.0
    return sum(1 for it in lst if it in tail_items) / len(lst)


# ปั่น Novelty: แทน i2 (Inception, 150 คน) ด้วย i12 (Spirited Away, 2 คน) — เปลี่ยนชิ้นเดียว
hacked_list = ["i1", "i4", "i7", "i10", "i12"]

# --------------------------- ตรวจคำตอบ ---------------------------
head, tail = my_head_tail_split(VIEWERS, 0.33)
total = sum(VIEWERS.values())
print(f"ปฏิสัมพันธ์รวม {total:,}  หนึ่งในสาม = {total/3:.2f}")
acc = 0
for it, c in sorted(VIEWERS.items(), key=lambda kv: -kv[1]):
    acc += c
    print(f"  {it:<5} ผู้เคยชม {c:>4}  สะสม {acc:>5}  สัดส่วนสะสม {acc/total:.4f}"
          f"   -> {'หัว' if it in head else 'หาง'}")

nov_x = np.mean([my_novelty(MODEL_X[u], VIEWERS, PLATFORM_USERS) for u in USERS8])
nov_y = np.mean([my_novelty(MODEL_Y[u], VIEWERS, PLATFORM_USERS) for u in USERS8])
print(f"\nNovelty@5 เฉลี่ย : Model X = {nov_x:.4f} บิต | Model Y = {nov_y:.4f} บิต\n")

check("9.1 กลุ่มหัวมี 2 เรื่อง (i1, i4)", head, {"i1", "i4"})
check("9.2 Novelty@5 ของ Model X (ทุกคนได้ชุดเดียวกัน)", nov_x, 1.0236)
check("9.3 Novelty@5 ของ Michael กับ Model Y",
      my_novelty(MODEL_Y["Michael"], VIEWERS, PLATFORM_USERS), 2.4879)
check("9.4 Novelty@5 เฉลี่ยของ Model Y", nov_y, 2.8525)
check("9.5 LongTail@5 เฉลี่ยของ Model X",
      np.mean([my_long_tail_ratio(MODEL_X[u], tail) for u in USERS8]), 0.6000)
check("9.6 LongTail@5 เฉลี่ยของ Model Y",
      np.mean([my_long_tail_ratio(MODEL_Y[u], tail) for u in USERS8]), 0.8250)

nov_hack = my_novelty(hacked_list, VIEWERS, PLATFORM_USERS)
check("9.7 ปั่น Novelty ขึ้น ≥ 40% โดยเปลี่ยนรายการเดียว",
      (nov_hack / nov_x >= 1.40) and (len(set(hacked_list) - set(MODEL_X["Emma"])) == 1), True)
print(f"\nลิสต์เดิม  {MODEL_X['Emma']} -> Novelty = {nov_x:.4f} บิต")
print(f"ลิสต์ปั่น  {hacked_list} -> Novelty = {nov_hack:.4f} บิต ({(nov_hack/nov_x-1)*100:+.1f}%)")

### 📝 ข้อ 9 กำลังสอนอะไร

**Novelty คือ Self-Information** $-\log_2 p(i)$ คือปริมาณสารสนเทศของการได้รู้จักรายการนั้น
รายการที่คนรู้จักน้อยมี $p(i)$ ต่ำ จึงให้ค่าสูง
ตีความว่า *"ระบบกำลังบอกสิ่งที่ผู้ใช้ไม่น่าจะรู้อยู่แล้วมากน้อยเพียงใด"*

Novelty = 1.0236 บิต ≈ สารสนเทศเท่ากับการรู้ผลโยนเหรียญหนึ่งครั้ง

**จุดที่พลาดกันบ่อยที่สุด — ฐานลอการิทึม**
ถ้าใช้ `np.log` (ฐาน $e$) แทน `np.log2` ค่าจะกลายเป็น 0.7095 แทน 1.0236 ทันที
(ต่างกันด้วยตัวประกอบ $\ln 2 \approx 0.693$) หน่วยเปลี่ยนจาก "บิต" เป็น "แนต"
→ **ต้องประกาศฐานลอการิทึมทุกครั้งที่รายงาน**

**เส้นแบ่งหัว–หางไม่มีเกณฑ์ตายตัว** ในข้อนี้ใช้เกณฑ์หนึ่งในสาม: ปฏิสัมพันธ์รวม 1,550
หนึ่งในสาม = 516.67 เมื่อไล่สะสม i1 (400) แล้ว i4 (300) รวมเป็น 700 ซึ่งเกิน 516.67
จึงได้กลุ่มหัวเพียง 2 เรื่องจาก 12 (16.7% ของคลัง) แต่ครองปฏิสัมพันธ์ 45.2%

เกณฑ์อื่นที่ใช้กัน: Top-2, 20% ของรายการ, หรือครอบคลุม 80% ของการรับชม
→ **ต้องประกาศวิธีแบ่งทุกครั้ง** เพราะเปลี่ยนเกณฑ์แล้ว LongTailRatio เปลี่ยนทันที

### ⚠️ ช่องโหว่ที่สำคัญที่สุดของ Novelty

เปลี่ยน **รายการเดียว** จาก Inception (150 คนเคยดู) เป็น Spirited Away (2 คนเคยดู)
ค่า Novelty พุ่งจาก 1.0236 เป็น 2.2693 บิต — **สูงขึ้น 122%**

| รายการ | ผู้เคยชม | $p(i)$ | $-\log_2 p(i)$ |
|---|---|---|---|
| Inception | 150 | 0.300 | 1.7370 |
| Spirited Away | 2 | 0.004 | 7.9658 |

รายการเดียวที่แทบไม่มีใครดู ดันค่าเฉลี่ยขึ้นได้มหาศาล เพราะ $-\log_2$ โตเร็วมากเมื่อ $p \to 0$

> **ข้อปฏิบัติ** ห้ามใช้ Novelty เป็น Primary Metric เดี่ยว ๆ
> ถ้าตั้งเป้าให้ระบบ "เพิ่ม Novelty" โดยไม่มี Guardrail ด้านความแม่นยำ
> แบบจำลองจะเรียนรู้ที่จะยัดของที่ไม่มีใครดูเข้าไปหนึ่งชิ้นทุกลิสต์ — และได้คะแนนเต็ม

---
## ข้อ 10 — Calibration Error และ Parity Gap ⭐⭐⭐
**หัวข้อ 2.5.5–2.5.6**

$$\text{CalibrationError}=\frac{1}{2}\sum_{c}\left|q(c)-p(c)\right|$$

* $p(c)$ = สัดส่วนแนว $c$ ใน **ประวัติ** ของผู้ใช้ (ใช้ `PROFILE8[u]` = เรื่องที่เขาให้คะแนน ≥ 4)
* $q(c)$ = สัดส่วนแนว $c$ ใน **รายการที่แนะนำ**

$$\text{ParityGap}=\max_g \overline{\text{metric}}_g-\min_g \overline{\text{metric}}_g$$

ใช้ `Precision@5` เป็นตัวชี้วัด และแบ่งกลุ่มด้วย `USER_GROUP` (ประวัติยาว / ประวัติสั้น)

> **กับดัก Leveling Down** ระบบที่แย่เท่ากันกับทุกคนจะได้ `ParityGap = 0` "สมบูรณ์แบบ"
> ขณะที่ระบบที่ดีกว่าใน *ทุกกลุ่ม* อาจได้ค่าสูงกว่า
> เกณฑ์ตายตัวแบบ "Gap ≤ 0.05" จะปฏิเสธระบบที่ดีกว่าทันที
> → **ต้องรายงานค่าเฉลี่ยรายกลุ่มควบคู่กับช่องว่างเสมอ**

In [ ]:
# ============================== เฉลยข้อ 10 ==============================
def genre_distribution(items, genre_of):
    c = Counter(genre_of.get(i, "?") for i in items)
    tot = sum(c.values())
    return {k: v / tot for k, v in c.items()} if tot else {}


def my_calibration_error(reclist, user_profile, genre_of):
    p = genre_distribution(list(user_profile), genre_of)     # ประวัติผู้ใช้
    q = genre_distribution(list(reclist), genre_of)          # รายการที่แนะนำ
    cats = set(p) | set(q)                                   # ต้องรวมแนวที่มีในฝั่งใดฝั่งหนึ่ง
    return 0.5 * sum(abs(q.get(c, 0.0) - p.get(c, 0.0)) for c in cats)


def my_parity_gap(scores_by_group):
    means = {g: float(np.mean(v)) for g, v in scores_by_group.items() if len(v)}
    if len(means) < 2:
        return 0.0
    return max(means.values()) - min(means.values())


def precision5(model_dict, user):
    return len(set(model_dict[user]) & RELEVANT8[user]) / K5


# --------------------------- ตรวจคำตอบ ---------------------------
print("โปรไฟล์แนวของ Sophia :", {k: round(v, 4) for k, v in
                                 genre_distribution(PROFILE8["Sophia"], GENRE).items()})
print("แนวที่ Model X แนะนำ  :", {k: round(v, 4) for k, v in
                                 genre_distribution(MODEL_X["Sophia"], GENRE).items()})
print("แนวที่ Model Y แนะนำ  :", {k: round(v, 4) for k, v in
                                 genre_distribution(MODEL_Y["Sophia"], GENRE).items()})
print()

check("10.1 สัดส่วนรวมเป็น 1.0",
      sum(genre_distribution(MODEL_X["Emma"], GENRE).values()), 1.0000)
check("10.2 CalibErr ของ Sophia กับ Model X",
      my_calibration_error(MODEL_X["Sophia"], PROFILE8["Sophia"], GENRE), 0.6000)
check("10.3 CalibErr ของ Sophia กับ Model Y",
      my_calibration_error(MODEL_Y["Sophia"], PROFILE8["Sophia"], GENRE), 0.0000)

pk_x = {u: precision5(MODEL_X, u) for u in USERS8}
pk_y = {u: precision5(MODEL_Y, u) for u in USERS8}
grp_x = {g: [pk_x[u] for u in us] for g, us in USER_GROUP.items()}
grp_y = {g: [pk_y[u] for u in us] for g, us in USER_GROUP.items()}

print(f"{'กลุ่ม':<14}{'P@5 ของ Model X':>20}{'P@5 ของ Model Y':>20}")
print("-" * 54)
for g in USER_GROUP:
    print(f"{g:<14}{np.mean(grp_x[g]):>20.4f}{np.mean(grp_y[g]):>20.4f}")
print(f"{'ช่องว่าง (Gap)':<14}{my_parity_gap(grp_x):>20.4f}{my_parity_gap(grp_y):>20.4f}\n")

check("10.4 P@5 ของ Emma กับ Model Y", pk_y["Emma"], 0.8000)
check("10.5 ParityGap ของ Model X", my_parity_gap(grp_x), 0.0000)
check("10.6 ParityGap ของ Model Y", my_parity_gap(grp_y), 0.2000)
check("10.7 P@5 เฉลี่ยของ Y สูงกว่า X ในทุกกลุ่ม",
      all(np.mean(grp_y[g]) > np.mean(grp_x[g]) for g in USER_GROUP), True)

### 📝 ข้อ 10 กำลังสอนอะไร

### Calibration Error

วัดว่า **สัดส่วนประเภทของรายการที่ระบบแนะนำ สอดคล้องกับความสนใจเดิมของผู้ใช้เพียงใด**
ใช้ระยะทางรวมแปรผัน (Total Variation Distance) ซึ่งมีตัวประกอบ $\tfrac{1}{2}$ อยู่ข้างหน้า
เพื่อให้ค่าอยู่ในช่วง $[0,1]$

**วิธีตีความที่ใช้ได้จริง** ค่า 0.20 กับลิสต์ 10 รายการ หมายถึง
*"ต้องปรับประมาณ 2 จาก 10 รายการ (จากแนวหนึ่งไปอีกแนวหนึ่ง) จึงจะได้สัดส่วนตรงกับประวัติ"*

ในข้อนี้ Sophia ชอบโรแมนติกกับแอนิเมชันเป็นหลัก Model Y แนะนำตรงสัดส่วนพอดี → 0.0000
ส่วน Model X ยัดไซไฟและแอ็กชันเข้ามา ซึ่งไม่มีในประวัติเลย → 0.6000

**จุดที่พลาดบ่อย** ลืมรวมแนวที่ปรากฏ *เฉพาะฝั่งใดฝั่งหนึ่ง*
ต้องใช้ `set(p) | set(q)` ไม่ใช่ `set(p) & set(q)` มิฉะนั้นแนวที่ระบบแนะนำ
แต่ไม่มีในประวัติจะหายไปจากผลรวม และค่าจะต่ำกว่าความจริง

### ⚠️ Parity Gap และกับดัก Leveling Down

| | P@5 ประวัติยาว | P@5 ประวัติสั้น | Parity Gap |
|---|---|---|---|
| Model X | 0.4000 | 0.4000 | **0.0000** ← "เป็นธรรมสมบูรณ์" |
| Model Y | 0.7500 | 0.5500 | **0.2000** |

**Model X ได้ช่องว่างเป็นศูนย์เป๊ะ** — ดูเหมือนเป็นธรรมที่สุดเท่าที่จะเป็นไปได้
แต่เป็นความเท่าเทียมที่เกิดจากการ **ให้บริการแย่พอ ๆ กันกับทุกคน**

ขณะที่ Model Y ดีกว่า **ในทุกกลุ่ม** (0.75 > 0.40 และ 0.55 > 0.40) แต่ได้ Gap สูงกว่า
เพราะมันยกระดับกลุ่มที่มีข้อมูลมากได้เร็วกว่า

**เกณฑ์ตายตัวแบบ "Gap ≤ 0.05" จะปฏิเสธ Model Y ทันที** และเลือก Model X
ซึ่งเป็นผลลัพธ์ที่กลับด้านจากความตั้งใจของนโยบายความเป็นธรรมโดยสิ้นเชิง

> **ข้อปฏิบัติ** ต้องรายงาน **ค่าเฉลี่ยของแต่ละกลุ่ม** ควบคู่กับช่องว่างเสมอ
> และต้องวัดความเป็นธรรมทั้งสองด้าน (User-side ด้วย Parity Gap และ Provider-side
> ด้วย Gini ของการเปิดเผยรายผู้ผลิต) — การวัดด้านเดียวแล้วอ้างว่าระบบเป็นธรรม
> เป็นข้อสรุปที่เกินหลักฐาน

---
## ข้อ 11 — Temporal Split และการตรวจการรั่วไหล ⭐⭐⭐
**หัวข้อ 2.8**

ใช้ชุดข้อมูล `INTERACTIONS` (มีคอลัมน์ `user_id`, `item_id`, `rating`, `timestamp`, `week`)

**สิ่งที่ต้องเขียน**

1. `my_global_temporal_split(df, train_frac)` — เรียงตามเวลา แล้วตัดที่จุดเดียวทั้งระบบ
2. `my_random_split(df, test_frac, seed)` — สุ่มโดยไม่สนใจเวลา (**ตัวอย่างของสิ่งที่ไม่ควรทำ**)
3. `my_leakage_report(train, test)` — นับเหตุการณ์ในชุดฝึกที่เกิด **หลัง** เหตุการณ์ที่เก่าที่สุดในชุดทดสอบ

**สิ่งที่จะได้เห็น** การแบ่งตามเวลาต้องได้ `n_future_in_train = 0` พอดี
ส่วนการแบ่งแบบสุ่มจะได้ค่าที่เกือบเท่ากับขนาดชุดฝึกทั้งหมด — ซึ่งเป็นไปไม่ได้ในระบบจริง

> **วินัยที่ป้องกันได้ข้อเดียว** การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ
> ก่อนคำนวณสถิติใด ๆ แม้แต่ค่าเฉลี่ยหรือพจนานุกรมรายการ

In [ ]:
# ============================== เฉลยข้อ 11 ==============================
def my_global_temporal_split(df, train_frac=0.8):
    d = df.sort_values("timestamp").reset_index(drop=True)   # เรียงตามเวลาก่อน
    cut = int(len(d) * train_frac)                           # จุดตัดจุดเดียวทั้งระบบ
    return d.iloc[:cut].copy(), d.iloc[cut:].copy()


def my_random_split(df, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    n_test = int(len(df) * test_frac)
    mask = np.zeros(len(df), dtype=bool)
    mask[idx[:n_test]] = True
    return df[~mask].copy(), df[mask].copy()


def my_leakage_report(train, test):
    if len(train) == 0 or len(test) == 0:
        return {"n_future_in_train": 0, "verdict": "ข้อมูลไม่พอตรวจ"}
    t0 = test["timestamp"].min()                             # เหตุการณ์ที่เก่าที่สุดของชุดทดสอบ
    n_future = int((train["timestamp"] > t0).sum())
    return {"n_train": len(train), "n_test": len(test),
            "n_future_in_train": n_future,
            "pct_future": 100.0 * n_future / len(train),
            "verdict": "ปลอดภัย" if n_future == 0 else "พบการรั่วไหล"}


# --------------------------- ตรวจคำตอบ ---------------------------
tr_t, te_t = my_global_temporal_split(INTERACTIONS, 0.8)
tr_r, te_r = my_random_split(INTERACTIONS, 0.2, seed=42)
rep_t = my_leakage_report(tr_t, te_t)
rep_r = my_leakage_report(tr_r, te_r)

print(f"{'วิธีแบ่ง':<26}{'ฝึก':>9}{'ทดสอบ':>9}{'อนาคตในชุดฝึก':>20}{'ผลตรวจ':>18}")
print("-" * 84)
for nm, tr, te, rp in [("ตามเวลาแบบรวม", tr_t, te_t, rep_t), ("สุ่ม (Random)", tr_r, te_r, rep_r)]:
    print(f"{nm:<26}{len(tr):>9,}{len(te):>9,}"
          f"{rp['n_future_in_train']:>13,} ({rp['pct_future']:>4.1f}%)"
          f"{rp['verdict']:>18}")

print(f"\nช่วงเวลาของชุดฝึก (temporal)  : สัปดาห์ {tr_t.week.min()} ถึง {tr_t.week.max()}")
print(f"ช่วงเวลาของชุดทดสอบ (temporal) : สัปดาห์ {te_t.week.min()} ถึง {te_t.week.max()}")
print(f"ช่วงเวลาของชุดฝึก (random)    : สัปดาห์ {tr_r.week.min()} ถึง {tr_r.week.max()}")
print(f"ช่วงเวลาของชุดทดสอบ (random)   : สัปดาห์ {te_r.week.min()} ถึง {te_r.week.max()}  <- ทับซ้อนกันเต็ม ๆ\n")

n_total = len(INTERACTIONS)
check("11.1 ขนาดชุดฝึกของ temporal split", len(tr_t), int(n_total * 0.8))
check("11.2 ชุดฝึก+ชุดทดสอบ = ข้อมูลทั้งหมด", len(tr_t) + len(te_t), n_total)
check("11.3 temporal split ไม่มีอนาคตในชุดฝึก", rep_t["n_future_in_train"], 0)
check("11.4 temporal split ได้ผลตรวจ 'ปลอดภัย'", rep_t["verdict"], "ปลอดภัย")
check("11.5 เวลาสูงสุดของชุดฝึก ≤ เวลาต่ำสุดของชุดทดสอบ",
      tr_t.timestamp.max() <= te_t.timestamp.min(), True)
check("11.6 random split พบการรั่วไหล", rep_r["verdict"], "พบการรั่วไหล")
check("11.7 random split รั่วเกิน 90% ของชุดฝึก",
      rep_r["n_future_in_train"] / len(tr_r) > 0.90, True)
check("11.8 random split ใช้ seed เดิมแล้วได้ผลเดิม",
      len(my_random_split(INTERACTIONS, 0.2, seed=42)[1]), len(te_r))

# ---- สาธิตการรั่วไหลจากสถิติ (statistic leakage)
print("\n" + "=" * 76)
print("โบนัส: การรั่วไหลจากสถิติ — ตัวอย่างที่ไม่มีบรรทัดใดดูผิดเลย")
print("=" * 76)
pop_wrong = Counter(INTERACTIONS.item_id)        # ❌ คำนวณจากข้อมูลทั้งหมดก่อนแบ่ง
pop_right = Counter(tr_t.item_id)                # ✓ คำนวณจากชุดฝึกเท่านั้น
top_wrong = [i for i, _ in pop_wrong.most_common(10)]
top_right = [i for i, _ in pop_right.most_common(10)]
overlap = len(set(top_wrong) & set(top_right))
print(f"  Top-10 ความนิยมจากข้อมูลทั้งหมด : {top_wrong}")
print(f"  Top-10 ความนิยมจากชุดฝึกเท่านั้น : {top_right}")
print(f"  ซ้ำกัน {overlap} จาก 10 รายการ -> ต่างกัน {10-overlap} รายการ")
print("  ทั้งสองบรรทัดเป็นโค้ดที่ 'ดูปกติ' ทั้งคู่ ต่างกันเพียงลำดับที่เขียน")
print("  วินัยเดียวที่ป้องกันได้: การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ")

### 📝 ข้อ 11 กำลังสอนอะไร

**ทำไม random split จึงรั่วเกือบ 100%**
`n_future_in_train` นับเหตุการณ์ในชุดฝึกที่เกิด *หลัง* เหตุการณ์ที่เก่าที่สุดของชุดทดสอบ
เมื่อสุ่ม 20% ไปเป็นชุดทดสอบโดยไม่สนเวลา เหตุการณ์ที่เก่าที่สุดของชุดทดสอบ
จะอยู่ใกล้จุดเริ่มต้นของข้อมูลเสมอ → แทบทุกแถวในชุดฝึกจึง "อยู่ในอนาคต" ของมัน

**สถานการณ์นี้เป็นไปไม่ได้ในระบบจริง** ระบบจริงถูกฝึกด้วยอดีตทั้งหมด
แล้วนำไปให้บริการอนาคต — ไม่มีทางที่แบบจำลองจะเห็นเหตุการณ์ของสัปดาห์หน้า

**กลไกที่ทำให้ตัวเลขเฟ้อ — Popularity Drift**
ความนิยมของรายการเลื่อนตามเวลาเสมอ (ภาพยนตร์ เพลง ข่าว สินค้า — ทุกโดเมน)
เมื่อแบ่งแบบสุ่ม แบบจำลอง TopPopular ได้เห็นความนิยม *ในสัปดาห์เดียวกับที่ต้องทำนาย*
จึงแม่นผิดปกติ ในตำราวัดได้ **+71.6%** โดยที่แบบจำลองไม่ได้ดีขึ้นเลยแม้แต่น้อย

### การรั่วไหลจากสถิติ (Statistic Leakage)

เซลล์โบนัสข้างบนสาธิตความผิดพลาดที่ **ผู้ตรวจอ่านโค้ดจับไม่ได้**

```python
# ❌ แบบผิด — คำนวณความนิยมก่อนแบ่งข้อมูล
pop = df.groupby('item_id').size()          # ใช้ข้อมูลทั้งหมด รวมชุดทดสอบ
train, test = global_temporal_split(df, 0.2)
model.fit(train, popularity=pop)             # ความนิยมมีอนาคตปนอยู่แล้ว

# ✓ แบบถูก — แบ่งก่อน แล้วจึงคำนวณสถิติจากชุดฝึกเท่านั้น
train, test = global_temporal_split(df, 0.2)
pop = train.groupby('item_id').size()
model.fit(train, popularity=pop)
```

ไม่มีบรรทัดใดที่ "ดูเหมือน" ผิด ไม่มี error ไม่มี warning
แต่ส่วนต่างระดับ 4–5% ที่เกิดขึ้นก็เพียงพอจะกลบผลของวิธีการใหม่ได้ทั้งหมด
เพราะงานตีพิมพ์จำนวนมากอ้างการปรับปรุงเพียง 2–5%

**รูปแบบการรั่วไหลอื่นที่ต้องระวัง** ปรับไฮเพอร์พารามิเตอร์บนชุดทดสอบ ·
ทำ k-core filter หลังแบ่งข้อมูล · normalize ด้วยค่าเฉลี่ยของทั้งชุด ·
สร้างพจนานุกรมรายการจากข้อมูลทั้งหมด

> **วินัยข้อเดียวที่ป้องกันได้ทุกกรณี** การแบ่งข้อมูลต้องเป็น **คำสั่งแรกเสมอ**
> ก่อนคำนวณสถิติใด ๆ แม้แต่ค่าเฉลี่ยหรือพจนานุกรมรายการ

---
## ข้อ 12 — Sampled Negatives และ Winner Flip ⭐⭐⭐
**หัวข้อ 2.9 — หัวข้อที่สำคัญที่สุดของทั้งบท**

เมื่อสุ่มรายการเชิงลบ $m$ ชิ้นจากคลังขนาด $|\mathcal{I}|$ อันดับที่สังเกตได้มีค่าคาดหวัง

$$\mathbb{E}[\tilde{\rho}]\approx 1+(\rho-1)\cdot\frac{m}{|\mathcal{I}|-1}$$

**สิ่งที่ต้องเขียน**

1. `my_ndcg_from_rank(rank, K)` — nDCG@K เมื่อมีของถูกเพียงหนึ่งชิ้นที่อันดับ `rank`
   (IDCG = 1 ดังนั้น nDCG = DCG และถ้า `rank > K` ต้องได้ 0 ทันที — **จุดตัดแข็ง**)
2. `expected_sampled_rank(true_rank, n_items, m)` — สูตรข้างบน
3. `my_sampled_rank(true_rank, n_items, m, rng)` — จำลองด้วยการแจกแจงไฮเพอร์จีออเมตริก
   (`rng.hypergeometric(ngood, nbad, nsample)`)

**สุดท้าย** หาค่า $m$ ที่ทำให้เกิด **Winner Flip** ระหว่างสองแบบจำลอง

* Model P — อันดับจริง = 12 สำหรับผู้ใช้ทุกคน (สม่ำเสมอ แต่หลุด Top-10 เสมอ)
* Model Q — อันดับ 3 สำหรับผู้ใช้ 30% และอันดับ 60 สำหรับ 70%

คลัง 500 รายการ ผู้ใช้ 800 คน K = 10 — **แบบจำลองใดดีกว่าสำหรับระบบที่แสดงสิบอันดับแรก**

In [ ]:
# ============================== เฉลยข้อ 12 ==============================
def my_ndcg_from_rank(rank, K=10):
    """IDCG = 1 (ของถูกชิ้นเดียวอยู่อันดับ 1) ดังนั้น nDCG = DCG
    ถ้า rank > K ได้ 0 ทันที — นี่คือ 'จุดตัดแข็ง' ที่ทำให้เกิด Winner Flip"""
    return 0.0 if rank > K else 1.0 / np.log2(rank + 1)


def expected_sampled_rank(true_rank, n_items, m):
    return 1 + (true_rank - 1) * m / (n_items - 1)


def my_sampled_rank(true_rank, n_items, m, rng):
    n_better = true_rank - 1               # ของที่เหนือกว่าเป้าหมาย
    n_neg = n_items - 1                    # รายการเชิงลบทั้งหมด
    drawn = rng.hypergeometric(ngood=n_better, nbad=n_neg - n_better,
                               nsample=min(m, n_neg))
    return 1 + int(drawn)


m_where_P_wins = 100        # ค่าที่ใช้แพร่หลายที่สุดในวรรณกรรม — และเป็นค่าที่ทำให้ P ชนะ

# --------------------------- ตรวจคำตอบ ---------------------------
check("12.1 อันดับ 1 -> nDCG@10 = 1.0", my_ndcg_from_rank(1, 10), 1.0000)
check("12.2 อันดับ 10 -> nDCG@10", my_ndcg_from_rank(10, 10), 0.2891)
check("12.3 อันดับ 11 -> nDCG@10 = 0 (จุดตัดแข็ง)", my_ndcg_from_rank(11, 10), 0.0000)
check("12.4 สูตรอันดับคาดหวัง (ρ=500, |I|=5000, m=99)",
      expected_sampled_rank(500, 5000, 99), 10.8822)

rng = np.random.default_rng(2569)
sim = np.array([my_sampled_rank(500, 5000, 99, rng) for _ in range(20000)])
check("12.5 การจำลองตรงกับทฤษฎี (คลาดเคลื่อน < 0.3)",
      abs(sim.mean() - expected_sampled_rank(500, 5000, 99)) < 0.3, True)
check("12.6 sampled nDCG@10 ของอันดับจริง 500 อยู่ราว 0.15",
      abs(np.mean([my_ndcg_from_rank(int(r), 10) for r in sim]) - 0.1482) < 0.02, True)

print(f"\nอันดับจริง 500 จากคลัง 5,000 : full nDCG@10 = {my_ndcg_from_rank(500, 10):.4f} (ศูนย์สนิท)")
print(f"สุ่มรายการเชิงลบ m = 99      : อันดับเฉลี่ย {sim.mean():.2f}  "
      f"เข้า Top-10 {(sim <= 10).mean()*100:.1f}%  "
      f"sampled nDCG@10 = {np.mean([my_ndcg_from_rank(int(r), 10) for r in sim]):.4f}")

# ---- Winner Flip
N_ITEMS, N_USERS, K = 500, 800, 10
ranks_P = np.full(N_USERS, 12)
ranks_Q = np.concatenate([np.full(int(0.30 * N_USERS), 3), np.full(N_USERS - int(0.30 * N_USERS), 60)])


def mean_sampled_ndcg(ranks, m, rng, n_items=N_ITEMS, K=K):
    if m is None:
        return float(np.mean([my_ndcg_from_rank(int(r), K) for r in ranks]))
    return float(np.mean([my_ndcg_from_rank(my_sampled_rank(int(r), n_items, m, rng), K)
                          for r in ranks]))


rng = np.random.default_rng(7)
print(f"\n{'m':>16}{'nDCG@10 ของ P':>18}{'nDCG@10 ของ Q':>18}{'ผู้ชนะ':>10}{'อัตราส่วน':>14}")
print("-" * 78)
for m in [20, 50, 100, 200, None]:
    p = mean_sampled_ndcg(ranks_P, m, rng)
    q = mean_sampled_ndcg(ranks_Q, m, rng)
    tag = "เต็มคลัง" if m is None else str(m)
    ratio = f"{p/q:.2f} เท่า" if q > 0 else "—"
    print(f"{tag:>16}{p:>18.4f}{q:>18.4f}{('P' if p > q else 'Q'):>10}{ratio:>14}")

p_full = mean_sampled_ndcg(ranks_P, None, rng)
q_full = mean_sampled_ndcg(ranks_Q, None, rng)
check("12.7 เต็มคลัง: P ได้ศูนย์สนิท", p_full, 0.0000)
check("12.8 เต็มคลัง: Q ชนะ P", q_full > p_full, True)
rng2 = np.random.default_rng(7)
p_m = mean_sampled_ndcg(ranks_P, m_where_P_wins, rng2)
q_m = mean_sampled_ndcg(ranks_Q, m_where_P_wins, rng2)
check(f"12.9 ที่ m = {m_where_P_wins} แล้ว P ชนะ Q", p_m > q_m, True)
print(f"\nที่ m = {m_where_P_wins} : P = {p_m:.4f} ชนะ Q = {q_m:.4f} ({p_m/q_m:.2f} เท่า)")
print(f"ที่เต็มคลัง   : P = {p_full:.4f} แพ้ Q = {q_full:.4f} อย่างสิ้นเชิง")

### 📝 ข้อ 12 กำลังสอนอะไร — หัวข้อที่สำคัญที่สุดของทั้งบท

### กลไก: การสุ่มบีบอันดับเข้าหา 1

เมื่อสุ่มรายการเชิงลบ $m$ ชิ้นจากคลังขนาด $|\mathcal{I}|$
ของที่ "เหนือกว่า" เป้าหมายมี $\rho - 1$ ชิ้น เมื่อสุ่มมา $m$ ชิ้น
จำนวนที่ยังเหนือกว่าเป็นตัวแปรสุ่ม **ไฮเพอร์จีออเมตริก** และ

$$\mathbb{E}[\tilde{\rho}] = 1+(\rho-1)\cdot\frac{m}{|\mathcal{I}|-1}$$

อันดับจริง 500 จากคลัง 5,000 → $1 + 499\times\frac{99}{4999} = 10.88$
และการจำลอง 20,000 รอบให้ 10.86 — **ทฤษฎีตรงกับการจำลอง**
จึงใช้สูตรนี้ประมาณผลกระทบล่วงหน้าได้โดยไม่ต้องรันการทดลอง

ผลคือ `nDCG@10` กระโดดจาก **0.0000 เป็น 0.1482** โดยที่แบบจำลอง
ไม่ได้เปลี่ยนพารามิเตอร์แม้แต่ตัวเดียว — สิ่งที่เปลี่ยนคือ **สนามสอบที่ง่ายลง 50 เท่า**

### ⚠️ ข้อคิดที่ผิด และเหตุผลที่ผิด

> *"ในเมื่อการสุ่มบีบทุกแบบจำลองเท่ากัน การเปรียบเทียบก็น่าจะยังยุติธรรม"*

**ผิด เพราะ Hard Cutoff ที่ K** — `nDCG@K` ไม่ใช่ฟังก์ชันเชิงเส้นของอันดับ

| อันดับ | nDCG@10 |
|---|---|
| 10 | 0.2891 |
| 11 | **0.0000** |

การประกอบ **ฟังก์ชันไม่เชิงเส้น** เข้ากับ **การแปลงเชิงเส้น** ไม่รักษาลำดับของค่าเฉลี่ย
(Krichene & Rendle, KDD 2020)

### Winner Flip ที่เกิดขึ้นจริง

| | Model P | Model Q |
|---|---|---|
| พฤติกรรม | อันดับ 12 สำหรับทุกคน | อันดับ 3 สำหรับ 30% · อันดับ 60 สำหรับ 70% |
| ที่ m = 100 | **ชนะ** ~1.76 เท่า | แพ้ |
| เต็มบัญชีรายการ | **0.0000 ศูนย์สนิท** | **ชนะ** |

**ผู้ชนะสลับกันโดยสิ้นเชิง** และไม่ใช่ความบังเอิญเชิงสถิติ แต่เป็นผลเชิงโครงสร้าง:
อันดับ 12 ของ P อยู่ *ใกล้* เส้นตัดมาก จึงถูกการสุ่ม "อุ้ม" เข้า Top-10 อย่างสม่ำเสมอ
ขณะที่อันดับ 60 ของผู้ใช้ 70% ของ Q ไกลเกินกว่าจะถูกอุ้ม

**ผลกระทบต่อทั้งสาขา** งานตีพิมพ์จำนวนมากที่ใช้ $m=100$ อาจสรุปผู้ชนะผิด
โดยที่การคำนวณทุกบรรทัดถูกต้อง

### ข้อกำหนดห้าข้อเมื่อจำเป็นต้องใช้การประเมินแบบสุ่ม

1. ระบุ $m$ และวิธีสุ่มให้ครบ (สม่ำเสมอ / ตามความนิยม / ในหมวดเดียวกัน — ให้ผลต่างกันมาก)
2. เรียกว่า **`sampled nDCG@K`** ไม่ใช่ `nDCG@K` เฉย ๆ
3. รายงานทั้งสองค่าหากทรัพยากรเอื้อ แล้วตรวจว่าอันดับของแบบจำลองตรงกันหรือไม่
4. **ห้ามเทียบค่าที่ $m$ ต่างกัน** และห้ามนำ sampled metric ของตนไปเทียบกับ full metric ของงานอื่น
5. $m < 50$ ถือเป็นการคัดกรองเบื้องต้นเท่านั้น ไม่ใช่ข้อสรุปที่ใช้ตัดสินได้

**ทางเลือกเมื่อคลังใหญ่เกินไป** แยกการประเมินเป็นสองขั้นให้ตรงกับสถาปัตยกรรมจริง:
Retrieval ด้วย ANN Index ดึงผู้สมัคร ~1,000–10,000 (วัด `Recall@1000`)
แล้ว Re-ranking (วัด `nDCG@10`) — เซตผู้สมัครมาจากกลไกเดียวกับที่ระบบจริงใช้ ไม่ใช่การสุ่ม

---
## ★ โจทย์ท้าทาย — สร้างแบบจำลองของตัวเอง ⭐⭐⭐⭐
**หัวข้อ 2.6 + 2.10 · ไม่มีเฉลยตายตัว ตรวจด้วยเกณฑ์ ไม่ใช่ด้วยค่าที่ถูกต้องเพียงค่าเดียว**

### โจทย์

ออกแบบฟังก์ชัน `my_ranker(user_id, candidates)` ที่คืนลิสต์รายการเรียงอันดับ
โดยต้องทำให้ผ่าน **ทั้งสามเงื่อนไข** พร้อมกัน เทียบกับเส้นฐาน `TopPopular`

| เงื่อนไข | เกณฑ์ |
|---|---|
| Primary Metric | `nDCG@10` **สูงกว่า** TopPopular |
| Guardrail 1 | `Coverage@10` **สูงกว่า** TopPopular |
| Guardrail 2 | `Novelty` **ไม่ต่ำกว่า** TopPopular |

### ข้อบังคับเชิงระเบียบวิธี

* ห้ามใช้ข้อมูลจาก `test` ในการสร้างแบบจำลองไม่ว่ากรณีใด (ทั้งความนิยม ค่าเฉลี่ย หรือรายชื่อรายการ)
* สถิติทุกตัวต้องคำนวณจาก `train` เท่านั้น
* ต้องรายงานผลพร้อม paired bootstrap ระดับผู้ใช้

### ใบ้

* เส้นฐานความนิยมล้วนจะกระจุกที่รายการเดิม ๆ — ลองผสมสัญญาณอื่นเข้าไป
* การ "ลงโทษ" รายการที่ถูกแนะนำไปแล้วหลายครั้ง ช่วยดัน Coverage ขึ้นได้
* ระวังไม่ให้ nDCG ตกจนแลกไม่คุ้ม — นี่คือการต่อรองบน Pareto Frontier ตัวจริง

In [ ]:
# ============================== เฉลยโจทย์ท้าทาย ==============================
import zlib

# สนามทดลอง: แบ่งตามเวลา 80/20 (ใช้ฟังก์ชันจากข้อ 11)
train_b, test_b = my_global_temporal_split(INTERACTIONS, 0.8)
all_items_b = sorted(INTERACTIONS.item_id.unique())
seen_b = train_b.groupby("user_id")["item_id"].apply(set).to_dict()
rel_b = test_b[test_b.rating >= 4].groupby("user_id")["item_id"].apply(set).to_dict()
pop_b = Counter(train_b.item_id)
pop_order_b = [it for it, _ in pop_b.most_common()]
n_users_b = train_b.user_id.nunique()


def baseline_toppopular(user_id, candidates):
    cs = set(candidates)
    head = [i for i in pop_order_b if i in cs]
    hs = set(head)
    return head + [i for i in candidates if i not in hs]


# ---------------------------------------------------------------------------
# แนวคิดของเฉลย — ผสมสองกลไกที่ตรงกับ "จุดอ่อน" ของ TopPopular ทีละข้อ
#
# กลไกที่ 1  ความนิยมถ่วงน้ำหนักตามเวลา (Recency-weighted popularity)
#            TopPopular นับทุกสัปดาห์เท่ากัน จึงตามความนิยมที่เลื่อนไม่ทัน
#            ให้น้ำหนัก exp(−Δสัปดาห์ / HALF) แก่เหตุการณ์ที่ใกล้จุดตัดมากกว่า
#            -> ยกทั้ง nDCG (ตรงกับ test ที่อยู่ถัดจากจุดตัด) และ Novelty
#
# กลไกที่ 2  การหมุนลิสต์รายผู้ใช้ (per-user rotation)
#            TopPopular ให้ลิสต์เดียวกันทุกคน Coverage จึงตันที่ K/|I|
#            ตรึงอันดับ 1..FIXED ไว้ (รักษาความแม่น) แล้วหมุนช่วง FIXED..FIXED+POOL
#            ด้วยค่าคงที่ที่ขึ้นกับ user_id -> ผู้ใช้ต่างคนเห็นชุดต่างกัน
#
# ข้อสำคัญเชิงระเบียบวิธี: ทุกสถิติคำนวณจาก train_b เท่านั้น ไม่แตะ test_b เลย
# ใช้ zlib.crc32 แทน hash() ของ Python เพราะ hash ของสตริงเปลี่ยนทุกครั้งที่รันใหม่
# (PYTHONHASHSEED) ซึ่งจะทำให้ผลทำซ้ำไม่ได้ — ผิดหลักการทำซ้ำได้ของบทนี้
# ---------------------------------------------------------------------------
HALF, FIXED, POOL = 4.0, 6, 12          # ไฮเพอร์พารามิเตอร์ (ควรจูนบนชุดตรวจสอบ ไม่ใช่ชุดทดสอบ)

_wmax = train_b.week.max()
_w = np.exp(-(_wmax - train_b.week) / HALF)
_rpop = train_b.assign(_w=_w).groupby("item_id")["_w"].sum().to_dict()
recency_order = sorted(all_items_b, key=lambda i: (-_rpop.get(i, 0.0), i))


def my_ranker(user_id, candidates):
    cs = set(candidates)
    pool = [i for i in recency_order if i in cs]
    fixed = pool[:FIXED]
    cand = pool[FIXED:FIXED + POOL]
    off = zlib.crc32(user_id.encode()) % max(1, len(cand))    # หมุนแบบกำหนดได้ ทำซ้ำได้
    return fixed + cand[off:] + cand[:off] + pool[FIXED + POOL:] + [i for i in candidates if i not in cs]


# --------------------------- ประเมินและตรวจเกณฑ์ ---------------------------
def bench(rank_fn, K=10):
    nd, lists, nov = [], [], []
    scores = {}
    for u, items in rel_b.items():
        cands = [i for i in all_items_b if i not in seen_b.get(u, set())]
        if not items & set(cands):
            continue
        top = rank_fn(u, cands)[:K]
        s = my_ndcg_at_k(top, {i: 1.0 for i in items}, K)
        nd.append(s); scores[u] = s; lists.append(top)
        nov.append(my_novelty(top, dict(pop_b), n_users_b))
    shown = {x for l in lists for x in l}
    return {"nDCG@10": float(np.mean(nd)),
            "Coverage@10": len(shown) / len(all_items_b),
            "Novelty": float(np.mean(nov))}, scores


base_res, base_scores = bench(baseline_toppopular)
mine_res, mine_scores = bench(my_ranker)
display(pd.DataFrame({"TopPopular (เส้นฐาน)": base_res, "RecencyRotate (เฉลย)": mine_res}).T.round(4))

c1 = mine_res["nDCG@10"] > base_res["nDCG@10"]
c2 = mine_res["Coverage@10"] > base_res["Coverage@10"]
c3 = mine_res["Novelty"] >= base_res["Novelty"]
print(f"\nPrimary  nDCG@10 สูงกว่าเส้นฐาน      : {'✅' if c1 else '❌'}"
      f"  ({(mine_res['nDCG@10']/base_res['nDCG@10']-1)*100:+.1f}%)")
print(f"Guard 1  Coverage@10 สูงกว่าเส้นฐาน  : {'✅' if c2 else '❌'}"
      f"  ({(mine_res['Coverage@10']/base_res['Coverage@10']-1)*100:+.1f}%)")
print(f"Guard 2  Novelty ไม่ต่ำกว่าเส้นฐาน    : {'✅' if c3 else '❌'}"
      f"  ({(mine_res['Novelty']/base_res['Novelty']-1)*100:+.1f}%)")
check("★ ผ่านครบทั้งสามเงื่อนไข", c1 and c2 and c3, True)

# ---- รายงานความไม่แน่นอน (paired bootstrap ระดับผู้ใช้)
common = sorted(set(base_scores) & set(mine_scores))
d = np.array([mine_scores[u] - base_scores[u] for u in common])
rngb = np.random.default_rng(11)
boots = d[rngb.integers(0, len(d), size=(4000, len(d)))].mean(axis=1)
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\nเปรียบเทียบบนผู้ใช้ร่วม {len(common):,} คน (สุ่มซ้ำระดับผู้ใช้ 4,000 รอบ)")
print(f"ส่วนต่างเฉลี่ยรายผู้ใช้ = {d.mean():+.4f}   CI 95% = [{lo:+.4f}, {hi:+.4f}]")
print(f"ผู้ใช้ที่ได้ผลแย่ลง = {(d < 0).mean()*100:.1f}%   "
      f"ข้อสรุป: {'มีนัยสำคัญ' if (lo > 0 or hi < 0) else 'ไม่มีหลักฐานว่าต่างกัน'}")

### 📝 โจทย์ท้าทายกำลังสอนอะไร

**นี่ไม่ใช่โจทย์ที่มีคำตอบเดียว** เฉลยข้างบนเป็นเพียงหนึ่งในหลายวิธีที่ผ่านเกณฑ์
สิ่งที่โจทย์ต้องการฝึกคือ **วิธีคิดเวลาต้องปรับ Primary Metric โดยไม่พัง Guardrail**

### ทำไมเฉลยนี้จึงได้ผล — ออกแบบจาก "จุดอ่อนที่รู้อยู่แล้ว" ของเส้นฐาน

| จุดอ่อนของ TopPopular | กลไกที่ใช้แก้ | ตัวชี้วัดที่ดีขึ้น |
|---|---|---|
| นับทุกสัปดาห์เท่ากัน จึงตามความนิยมที่เลื่อนไม่ทัน | ถ่วงน้ำหนักตามเวลา $e^{-\Delta w/\text{HALF}}$ | nDCG@10 · Novelty |
| ให้ลิสต์เดียวกันทุกคน Coverage จึงตันที่ $K/|\mathcal{I}|$ | หมุนช่วงกลางของลิสต์รายผู้ใช้ | Coverage@10 |

**ข้อสังเกตสำคัญ** ชุดข้อมูล `INTERACTIONS` ของแบบฝึกหัดถูกสร้างจากความนิยมล้วน ๆ
ไม่มีโครงสร้างความชอบรายบุคคลเลย ดังนั้น **การทำ personalization แบบ collaborative filtering
จะไม่ช่วยอะไรเลยในชุดนี้** — สิ่งเดียวที่ยังมีสัญญาณเหลืออยู่คือ *เวลา* (popularity drift)

นี่เองคือบทเรียนที่ซ่อนอยู่: **ก่อนจะเลือกอัลกอริทึม ต้องรู้ก่อนว่าข้อมูลมีสัญญาณอะไรให้ใช้**
การยัด MF-SVD เข้าไปกับข้อมูลที่ไม่มีโครงสร้างส่วนบุคคล ก็ไม่ต่างจากการเดา

### สามกับดักที่ต้องหลบให้ได้

**1. การรั่วไหล** ถ้าเผลอคำนวณความนิยมจาก `INTERACTIONS` ทั้งก้อน (ไม่ใช่ `train_b`)
คะแนนจะพุ่งขึ้นทันที และไม่มี error ใด ๆ ปรากฏ — แต่ผลที่ได้ไร้ความหมาย

**2. การทำซ้ำไม่ได้** `hash("u001")` ของ Python เปลี่ยนค่าทุกครั้งที่รันโปรเซสใหม่
(เพราะ `PYTHONHASHSEED` สุ่ม) ถ้าใช้มันทำ rotation ผลจะไม่เหมือนเดิมเลยในการรันครั้งถัดไป
เฉลยจึงใช้ `zlib.crc32` ซึ่งเป็นฟังก์ชันกำหนดได้

**3. การจูนบนชุดทดสอบ** `HALF`, `FIXED`, `POOL` คือไฮเพอร์พารามิเตอร์
ในงานจริงต้องจูนบน **ชุดตรวจสอบ (validation)** ที่กันไว้จากชุดฝึก
การไล่ลองค่าจนกว่าตัวเลขบนชุดทดสอบจะสวย คือการรั่วไหลอีกรูปแบบหนึ่ง
และเป็นรูปแบบที่พบบ่อยที่สุดในงานที่ทำซ้ำไม่ได้

### สิ่งที่ต้องเขียนในรายงาน

การได้ ✅ ครบสามข้อยังไม่พอ ต้องรายงานด้วยว่า

* **ส่วนต่างมีนัยสำคัญหรือไม่** — paired bootstrap ระดับผู้ใช้ พร้อมช่วงความเชื่อมั่น
* **ผู้ใช้กี่เปอร์เซ็นต์ได้ผลแย่ลง** — ค่าเฉลี่ยที่ดีขึ้นอาจปิดบังว่าคนจำนวนมากแย่ลง
* **ไฮเพอร์พารามิเตอร์ทุกตัวและวิธีเลือกมัน** — ไม่เช่นนั้นไม่มีใครทำซ้ำได้

> และสุดท้าย ผลออฟไลน์ทั้งหมดนี้เป็นเพียง **การคัดกรอง** เท่านั้น
> คะแนน `nDCG@10` ที่สูงขึ้นไม่ได้รับประกันความพึงพอใจของผู้ใช้จริง
> ต้องยืนยันด้วย A/B Test พร้อม Guardrail เชิงธุรกิจก่อนขึ้นใช้งานเสมอ

---
## สรุปคะแนน

รันเซลล์ถัดไปเพื่อดูว่าผ่านกี่ข้อ และข้อใดยังไม่ผ่าน

In [ ]:
show_score()
print("\nถ้าเฉลยนี้ผ่านครบทุกข้อ แปลว่าโค้ดทุกบรรทัดในสมุดเล่มนี้ตรวจสอบตัวเองได้")

---
## เฉลยคำถามทบทวนท้ายสมุดแบบฝึกหัด

**1. ทีม ก รายงาน `Precision@10` ทีม ข รายงาน `MRR` แล้วสรุปคนละอย่าง — จะป้องกันอย่างไร**

ประกาศ **Primary Metric เพียงตัวเดียวไว้ก่อนเห็นผล** และการประกาศต้องครบทั้งสี่ส่วน:
ชื่อตัวชี้วัด · ค่า K · วิธีเฉลี่ยข้ามผู้ใช้ · Candidate Set ที่ใช้
เช่น *"nDCG@10 เฉลี่ยแบบ macro-average ข้ามผู้ใช้ บน Full-Catalog Ranking
โดยตัดรายการที่ผู้ใช้เคยมีปฏิสัมพันธ์ในชุดฝึกออกก่อน"*
ตัวชี้วัดที่เหลือรายงานเป็น **Guardrail** พร้อมเกณฑ์ตัวเลข — ใช้ดูว่าอะไรพัง ไม่ใช้ตัดสินผู้ชนะ

**2. nDCG@10 สูงขึ้น 13.9% แต่เวลาใช้งานลดลง 2.1% — Guardrail ตัวใดจะจับได้**

`ILD` (Intra-List Diversity) — ในกรณีศึกษา 2.1 ของตำรา ILD ตกจาก 0.62 เหลือ 0.41
เพราะแบบจำลองใหม่แนะนำสิ่งที่คล้ายกับที่ผู้ใช้เพิ่งดู ซึ่งตรงกับข้อมูลอดีตจึงได้คะแนนออฟไลน์สูง
แต่ผู้ใช้จริงรู้สึกว่า "ซ้ำซาก" ถ้าประกาศเกณฑ์ `ILD ≥ 0.55` ไว้ล่วงหน้า
แบบจำลองนี้จะถูกปฏิเสธตั้งแต่ขั้นออฟไลน์ — **ก่อน** จะเสียเวลาและความไว้วางใจของผู้ใช้ไปกับ A/B Test

**3. "ใช้ m = 100 เหมือนงานอื่นก็ได้ จะได้เทียบกันได้" — ผิดตรงไหน**

ผิดสองชั้น

* **ชั้นที่หนึ่ง** ค่า sampled metric ขึ้นกับ *ขนาดคลัง* ด้วย ไม่ใช่แค่ $m$
  สูตร $\mathbb{E}[\tilde\rho] = 1+(\rho-1)\frac{m}{|\mathcal{I}|-1}$ มี $|\mathcal{I}|$ อยู่ในตัวส่วน
  คลัง 5,000 กับคลัง 500,000 ที่ $m=100$ เท่ากัน ให้ผลคนละสเกลโดยสิ้นเชิง
* **ชั้นที่สอง** *วิธีสุ่ม* ก็ต่างกันได้ (สม่ำเสมอ / ตามความนิยม / ในหมวดเดียวกัน)
  และให้ผลต่างกันมาก งานที่ระบุแค่ "m = 100" จึงยังทำซ้ำไม่ได้อยู่ดี

ที่สำคัญที่สุด ข้อ 12 แสดงแล้วว่าการสุ่ม **เปลี่ยนผู้ชนะได้** การเทียบข้ามงาน
จึงอาจให้ข้อสรุปที่กลับด้านจากความจริง ทั้งที่ทุกฝ่ายคำนวณถูกทุกบรรทัด

**4. A ดีกว่า B โดยเฉลี่ย แต่ผู้ใช้ 40% แย่ลง — ควรเปลี่ยนระบบหรือไม่**

**ไม่มีคำตอบทางเทคนิคเพียงคำตอบเดียว — นี่คือการตัดสินใจเชิงนโยบาย** สิ่งที่ต้องพิจารณา

* 40% ที่แย่ลงเป็นใคร ถ้าเป็นผู้ใช้ใหม่หรือกลุ่มที่มีข้อมูลน้อย การเปลี่ยนอาจทำลายการเติบโต
* แย่ลงมากแค่ไหน ผู้ใช้ 40% ที่แย่ลงนิดเดียว ต่างจาก 40% ที่แย่ลงจนเลิกใช้
* ต้นทุนของการสูญเสียผู้ใช้หนึ่งคน เทียบกับกำไรจากการที่อีก 60% ได้ผลดีขึ้น

**สิ่งที่ทำไม่ได้คือรายงานเพียงค่าเฉลี่ยแล้วบอกว่า "A ดีกว่า"** เพราะปิดบังข้อมูลที่ผู้ตัดสินใจต้องใช้

**5. Coverage@10 ชนเพดานที่ 1.0 แล้ว ยังบอกอะไรได้อีกหรือไม่**

**ไม่ได้อีกแล้ว** — นี่คือจุดบอดของ Coverage: เมื่อชนเพดานจะหยุดแยกความต่างทันที
ระบบที่แนะนำทุกรายการอย่างละครั้ง กับระบบที่แนะนำรายการหนึ่งเป็นพันครั้ง
และที่เหลืออย่างละครั้ง **ได้ Coverage = 1.0 เท่ากัน**

ต้องเปลี่ยนไปดู **Gini ของการเปิดเผย** (ซึ่งยังแยกได้) และแนบ **Lorenz Curve** ไปด้วยเสมอ
เช่นเดียวกับที่ต้องแนบ ROC Curve คู่กับ AUC เพราะค่าตัวเดียวสรุปได้เพียง "พื้นที่รวม"

---

## สรุปสิบข้อที่ต้องจำจากทั้งบท

1. MAE กับ RMSE ต้องรายงานคู่กัน — ช่องว่างระหว่างสองค่าคือสัญญาณของความไม่สม่ำเสมอ
2. ตัวชี้วัดที่ไม่ไวต่อตำแหน่งแยกแบบจำลองที่วางของถูกคนละตำแหน่งไม่ออก
3. การเลือกตัวชี้วัดคือการเลือกข้อสรุป — ประกาศ Primary Metric ก่อนเห็นผล
4. การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ ก่อนคำนวณสถิติใด ๆ
5. การสุ่มรายการเชิงลบเปลี่ยนผู้ชนะได้ ไม่ใช่แค่เปลี่ยนตัวเลข
6. ความแม่นยำเพียงด้านเดียวไม่พอ — ต้องมี Guardrail ที่ตรวจด้วยโปรแกรมได้
7. ตัวชี้วัดทุกตัวมีจุดบอด และต้องรู้ว่าจุดบอดนั้นอยู่ตรงไหน
8. ส่วนต่างที่เล็กกว่าความผันผวนจาก Seed ไม่ใช่การปรับปรุง
9. หน่วยของการสุ่มซ้ำคือ "ผู้ใช้" ไม่ใช่ "เหตุการณ์"
10. คะแนนออฟไลน์สูงไม่รับประกันความพึงพอใจ — ต้องยืนยันด้วย A/B Test

> **ประโยคปิดท้าย**
> คะแนนประเมินไม่ได้เป็นคุณสมบัติของแบบจำลองเพียงอย่างเดียว
> แต่เป็นผลร่วมของ **Model · ข้อมูล · ตัวชี้วัด · เวลา · Candidate Set · และโปรโตคอลการประเมินทั้งหมด**